# Single File Sky Calibration

**by Josh Dillon**, last updated August 17, 2026

This notebook is designed to infer as much information about the array from a single file as possible. It is patterned closely on `file_calibration.ipynb`, but replaces redundant-baseline calibration and absolute calibration with direct sky-model-based calibration (`hera_cal.skycal`). It also measures per-SNAP, per-X-engine-block signal loss ("decoherence") from the spectral structure of the gains and writes it to a sidecar file for downstream correction of inter-SNAP cross-correlations. It also includes RFI detection based on DPSS filtering of redundantly averaged crosses, patterned after Round 2 flagging in H6C IDR3.

Configuration comes from a TOML file (e.g. `hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the `TOML_FILE` environment variable; if none is given, the default settings in the cells below are used. Environment variables otherwise carry only paths and wrapper-level toggles.

When `SAVE_RESULTS` is `TRUE`, files with the following default names are written alongside the input, with names derived from `SUM_FILE` by replacing `.uvh5`:

* `*.sky.calfits` — per-antenna gain solutions, flags, and per-antenna sky-model $\chi^2$ waterfalls; any antenna identity relabelings are recorded in the `RELABELS` extra keyword
* `*.snap_decoherence.h5` — `SNAPDecoherence` sidecar with per-(SNAP, time, X-engine block) loss fractions and the antenna → SNAP mapping *as used here* (including identity repairs), for downstream correction of inter-SNAP cross-correlations
* `*.red_avg_zscore.h5` — `UVFlag` metrics file of cross-based RFI z-scores, for future whole-night synthesis
* `*.ant_class.csv` — machine-readable version of Table 1: per-antenna metric values and classifications
* `*.ant_metrics.hdf5` — `ant_metrics` results (only written when `USE_DIFF` is `TRUE`)

A fully-flagged file still produces all of these (as placeholders: unit gains, an empty sidecar, all-nan z-scores), so downstream, an absent file always means a failed job.

Here's a set of links to skip to particular figures and tables:

• [Figure 1: RFI Flagging](#Figure-1:-RFI-Flagging)

• [Figure 2: Plot of autocorrelations with classifications](#Figure-2:-Plot-of-autocorrelations-with-classifications)

• [Figure 3: Summary of antenna classifications prior to calibration](#Figure-3:-Summary-of-antenna-classifications-prior-to-calibration)

• [Figure 4: Array nodes and antenna identities](#Figure-4:-Array-nodes-and-antenna-identities)

• [Figure 5: Sky calibration goodness-of-fit and convergence](#Figure-5:-Sky-calibration-goodness-of-fit-and-convergence)

• [Figure 6: Redundantly-averaged calibrated data compared to the sky model](#Figure-6:-Redundantly-averaged-calibrated-data-compared-to-the-sky-model)

• [Figure 7: Delay-filtered redundantly-averaged z-scores before and after cross-based flagging](#Figure-7:-Delay-filtered-redundantly-averaged-z-scores-before-and-after-cross-based-flagging)

• [Figure 8: Relative Phase Calibration](#Figure-8:-Relative-Phase-Calibration)

• [Figure 9: Per-SNAP, per-X-engine-block decoherence](#Figure-9:-Per-SNAP,-per-X-engine-block-decoherence)

• [Figure 10: Gain spectra of the most decoherent SNAPs](#Figure-10:-Gain-spectra-of-the-most-decoherent-SNAPs)

• [Figure 11: Sky-model chi^2 per antenna across the array](#Figure-11:-Sky-model-chi^2-per-antenna-across-the-array)

• [Figure 12: Redundant-baseline chi^2 per antenna across the array](#Figure-12:-Redundant-baseline-chi^2-per-antenna-across-the-array)

• [Figure 13: Summary of antenna classifications after sky calibration](#Figure-13:-Summary-of-antenna-classifications-after-sky-calibration)

• [Table 1: Complete summary of per antenna classifications](#Table-1:-Complete-summary-of-per-antenna-classifications)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import toml
import numpy as np
from scipy import constants, interpolate
import copy
import glob
import json
import re
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', 1000)
from uvtools.plot import plot_antclass
from hera_qm import ant_metrics, ant_class, xrfi
from hera_cal import io, utils, redcal, datacontainer, abscal, skycal, noise, apply_cal
from hera_filters import dspec
from pyuvdata import UVFlag, UVData
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
_ = np.seterr(all='ignore')  # get rid of red warnings
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [ ]:
# this enables better memory management on linux
import ctypes
def malloc_trim():
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0) 
    except OSError:
        pass

## Parse inputs and outputs

To use this notebook interactively, you will have to provide a sum filename path if none exists as an environment variable. All other parameters have reasonable default values.

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"

# get infile names
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.38700.sum.uvh5' # If SUM_FILE is not defined in the environment variables, define it here.
TOML_FILE = os.environ.get("TOML_FILE", None)  # None --> use the default settings in the cells below
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'

JD = float(re.search(r'zen\.(\d+\.\d+)', SUM_FILE).group(1))
USE_DIFF = os.environ.get("USE_DIFF", (JD < 2460039))  # diffs were only recorded through H6C

# get outfilenames
AM_FILE = (SUM_FILE.replace('.uvh5', '.ant_metrics.hdf5') if SAVE_RESULTS else None)
ANTCLASS_FILE = (SUM_FILE.replace('.uvh5', '.ant_class.csv') if SAVE_RESULTS else None)
SKY_CAL_FILE = (SUM_FILE.replace('.uvh5', '.sky.calfits') if SAVE_RESULTS else None)
DECOHERENCE_FILE = (SUM_FILE.replace('.uvh5', '.snap_decoherence.h5') if SAVE_RESULTS else None)
RED_AVG_ZSCORE_FILE = (SUM_FILE.replace('.uvh5', '.red_avg_zscore.h5') if SAVE_RESULTS else None)

for fname in ['SUM_FILE', 'TOML_FILE', 'AM_FILE', 'ANTCLASS_FILE', 'SKY_CAL_FILE',
              'DECOHERENCE_FILE', 'RED_AVG_ZSCORE_FILE', 'SAVE_RESULTS']:
    print(f"{fname} = '{eval(fname)}'")

### Settings: defaults, then TOML overrides

Overridden below by the TOML's [GLOBAL_OPTS] and [FILE_SKY_CAL_OPTS] sections (if given)

In [ ]:
# sky calibration settings
SKY_MODEL_FILES_GLOB = None
SKYCAL_MIN_BL_LEN = 1.0    # in meters
SKYCAL_MAX_BL_LEN = 120.0  # in meters
SC_MAXITER = 100
SC_MAX_RERUN = 10
SC_MAX_DIVERGENT_CHANS = 25
MAX_BAND_FLAG_FRAC = 0.5
SC_MAX_CHISQ_FLAGGING_DYNAMIC_RANGE = 1.5
CALIBRATE_CROSS_POLS = True
AUTO_REPAIR_IDENTITIES = True
SKIP_OUTRIGGERS = True

# FM radio band (MHz)
FM_LOW_FREQ = 87.5
FM_HIGH_FREQ = 108.0

# cross-based RFI flagging settings (defaults match delay_filtered_average_zscore)
CROSS_RFI_MIN_SAMP_FRAC = .15
CROSS_RFI_FILTER_DELAY = 750  # in ns
CROSS_RFI_Z_THRESH = 4
CROSS_RFI_WS_Z_THRESH = 2

# auto-based RFI settings
RFI_DPSS_HALFWIDTH = 300e-9  # in s
RFI_NSIG = 4

# load TOML overrides, injecting each key as an uppercase global
toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})
for toml_section in ['GLOBAL_OPTS', 'FILE_SKY_CAL_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val

# diff files, where available, enable classifiers that need noise realizations;
# without them, RTP's antenna classification is loaded instead
DIFF_FILE = (SUM_FILE.replace('sum', 'diff') if USE_DIFF else None)
RTP_ANTCLASS = (None if USE_DIFF else SUM_FILE.replace('.uvh5', '.rtp_ant_class.csv'))

# print settings
for setting in ['JD', 'USE_DIFF', 'DIFF_FILE', 'RTP_ANTCLASS', 'SKY_MODEL_FILES_GLOB',
                'SKYCAL_MIN_BL_LEN', 'SKYCAL_MAX_BL_LEN', 'SC_MAXITER', 'SC_MAX_RERUN',
                'SC_MAX_CHISQ_FLAGGING_DYNAMIC_RANGE', 'CALIBRATE_CROSS_POLS', 'AUTO_REPAIR_IDENTITIES',
                'SKIP_OUTRIGGERS', 'FM_LOW_FREQ', 'FM_HIGH_FREQ', 'CROSS_RFI_MIN_SAMP_FRAC',
                'CROSS_RFI_FILTER_DELAY', 'CROSS_RFI_Z_THRESH', 'CROSS_RFI_WS_Z_THRESH',
                'RFI_DPSS_HALFWIDTH', 'RFI_NSIG']:
    print(f'{setting} = {eval(setting)}')

### Antenna classification bounds: defaults, then TOML overrides

In [ ]:
# Default classification bound scalars, overridden below by the TOML's [ANT_CLASS_BOUNDS] section (if given)
AM_CORR_BAD = 0.35
AM_CORR_SUSPECT = 0.45
AM_XPOL_BAD = -0.1
AM_XPOL_SUSPECT = 0.0
SUSPECT_SOLAR_ALTITUDE = -4.0
BAD_SOLAR_ALTITUDE = -2.0  # in degrees, conservative
MAX_ZEROS_PER_EO_SPEC_GOOD = 2
MAX_ZEROS_PER_EO_SPEC_SUSPECT = 8
AUTO_POWER_GOOD_LOW = 5.0
AUTO_POWER_GOOD_HIGH = 30.0
AUTO_POWER_SUSPECT_LOW = 1.0
AUTO_POWER_SUSPECT_HIGH = 60.0
AUTO_SLOPE_GOOD_LOW = -0.4
AUTO_SLOPE_GOOD_HIGH = 0.4
AUTO_SLOPE_SUSPECT_LOW = -0.6
AUTO_SLOPE_SUSPECT_HIGH = 0.6
AUTO_RFI_GOOD = 1.5
AUTO_RFI_SUSPECT = 2.0
AUTO_SHAPE_GOOD = 0.1
AUTO_SHAPE_SUSPECT = 0.2
BAD_XENGINE_ZCUT = 10.0
IDENTITY_COHERENCE_GOOD = 0.75
IDENTITY_COHERENCE_SUSPECT = 0.5
IDENTITY_REPAIR_MARGIN = 0.2
SC_CSPA_GOOD = 2.0
SC_CSPA_SUSPECT = 3.0
DECO_MAX_SUSPECT = 0.05
DECO_MAX_BAD = 0.10
DECO_JUMP_SUSPECT = 0.025
DECO_JUMP_BAD = 0.05

if 'ANT_CLASS_BOUNDS' in toml_options:
    print(f'Loading overrides from [ANT_CLASS_BOUNDS] in {TOML_FILE}.')
    for key, val in toml_options['ANT_CLASS_BOUNDS'].items():
        globals()[key.upper()] = val

# ant_metrics bounds for low correlation / dead antennas
am_corr_bad = (0, AM_CORR_BAD)
am_corr_suspect = (AM_CORR_BAD, AM_CORR_SUSPECT)

# ant_metrics bounds for cross-polarized antennas
am_xpol_bad = (-1, AM_XPOL_BAD)
am_xpol_suspect = (AM_XPOL_BAD, AM_XPOL_SUSPECT)

# bounds on solar altitude (in degrees)
good_solar_altitude = (-90, SUSPECT_SOLAR_ALTITUDE)
suspect_solar_altitude = (SUSPECT_SOLAR_ALTITUDE, BAD_SOLAR_ALTITUDE)
bad_solar_altitude = (BAD_SOLAR_ALTITUDE, 90)

# bounds on zeros in spectra
good_zeros_per_eo_spectrum = (0, MAX_ZEROS_PER_EO_SPEC_GOOD)
suspect_zeros_per_eo_spectrum = (0, MAX_ZEROS_PER_EO_SPEC_SUSPECT)

# bounds on autocorrelation power
auto_power_good = (AUTO_POWER_GOOD_LOW, AUTO_POWER_GOOD_HIGH)
auto_power_suspect = (AUTO_POWER_SUSPECT_LOW, AUTO_POWER_SUSPECT_HIGH)

# bounds on autocorrelation slope
auto_slope_good = (AUTO_SLOPE_GOOD_LOW, AUTO_SLOPE_GOOD_HIGH)
auto_slope_suspect = (AUTO_SLOPE_SUSPECT_LOW, AUTO_SLOPE_SUSPECT_HIGH)

# bounds on autocorrelation RFI
auto_rfi_good = (0, AUTO_RFI_GOOD)
auto_rfi_suspect = (0, AUTO_RFI_SUSPECT)

# bounds on autocorrelation shape
auto_shape_good = (0, AUTO_SHAPE_GOOD)
auto_shape_suspect = (0, AUTO_SHAPE_SUSPECT)

# bound on per-xengine non-noiselike power in diff
bad_xengine_zcut = BAD_XENGINE_ZCUT

# bounds on the identity audit's delay-searched data/model phase coherence, and
# the margin required before a mislabeled antenna is automatically relabeled (the
# good bound is also the minimum winning-candidate coherence for a relabeling)
identity_coherence_good = (IDENTITY_COHERENCE_GOOD, 1)
identity_coherence_suspect = (IDENTITY_COHERENCE_SUSPECT, 1)
identity_repair_margin = IDENTITY_REPAIR_MARGIN

# bounds on normalized chi^2 per antenna vs the sky model
sc_cspa_good = (0, SC_CSPA_GOOD)
sc_cspa_suspect = (0, SC_CSPA_SUSPECT)

# bounds on the decoherence of each antenna's SNAP: an absolute limit on the worst loss
# fraction p, and a limit on the largest change in p between adjacent X-engine blocks
# within a band (p is relative to each band's cleanest block, so changes across the
# band split at FM are not counted)
deco_max_good = (0, DECO_MAX_SUSPECT)
deco_max_suspect = (0, DECO_MAX_BAD)
deco_jump_good = (0, DECO_JUMP_SUSPECT)
deco_jump_suspect = (0, DECO_JUMP_BAD)

# print bounds
for bound in ['am_corr_bad', 'am_corr_suspect', 'am_xpol_bad', 'am_xpol_suspect',
              'good_solar_altitude', 'suspect_solar_altitude',
              'good_zeros_per_eo_spectrum', 'suspect_zeros_per_eo_spectrum',
              'auto_power_good', 'auto_power_suspect', 'auto_slope_good', 'auto_slope_suspect',
              'auto_rfi_good', 'auto_rfi_suspect', 'auto_shape_good', 'auto_shape_suspect',
              'bad_xengine_zcut', 'identity_coherence_good', 'identity_coherence_suspect',
              'identity_repair_margin', 'sc_cspa_good', 'sc_cspa_suspect',
              'deco_max_good', 'deco_max_suspect', 'deco_jump_good', 'deco_jump_suspect']:
    print(f'{bound} = {eval(bound)}')

## Load sum and diff data

In [ ]:
read_start = time.time()
hd = io.HERADataFastReader(SUM_FILE)
data, _, _ = hd.read(read_flags=False, read_nsamples=False)
if USE_DIFF:
    hd_diff = io.HERADataFastReader(DIFF_FILE)
    diff_data, _, _ = hd_diff.read(read_flags=False, read_nsamples=False, dtype=np.complex64, fix_autos_func=np.real)
print(f'Finished loading data in {(time.time() - read_start) / 60:.2f} minutes.')

In [ ]:
ants = sorted(set([ant for bl in hd.bls for ant in utils.split_bl(bl)]))
auto_bls = [bl for bl in data if (bl[0] == bl[1]) and (utils.split_pol(bl[2])[0] == utils.split_pol(bl[2])[1])]
antpols = sorted(set([ant[1] for ant in ants]))

In [ ]:
# print basic information about the file
print(f'File: {SUM_FILE}')
print(f'JDs: {hd.times} ({np.median(np.diff(hd.times)) * 24 * 3600:.5f} s integrations)')
print(f'LSTS: {hd.lsts * 12 / np.pi } hours')
print(f'Frequencies: {len(hd.freqs)} {np.median(np.diff(hd.freqs)) / 1e6:.5f} MHz channels from {hd.freqs[0] / 1e6:.5f} to {hd.freqs[-1] / 1e6:.5f} MHz')
print(f'Antennas: {len(hd.data_ants)}')
print(f'Polarizations: {hd.pols}')

## Classify good, suspect, and bad antpols

In [ ]:
ALL_FLAGGED = False
def all_flagged():
    if ALL_FLAGGED:
        print('All antennas are flagged, so this cell is being skipped.')
    return ALL_FLAGGED

# initialize classes and results to None to help make Table 1 when everything is flagged
overall_class = None
am_totally_dead = None
am_corr = None
am_xpol = None
solar_class = None
zeros_class = None
auto_power_class = None
auto_slope_class = None
auto_rfi_class = None
auto_shape_class = None
xengine_diff_class = None
meta = None
cspa = None
total_chisq = None
skycal_class = None
deco_class = None
deco_jump_class = None
identity_class = None
ant_snap = None
self_coherence = None
labeled_to_true = {}
identity_repair_note = ''
sd = None
dmeta = None
zscore = None
cross_rfi_flags = None
redcal_chisq = None
redcal_cspa = None
total_chisq = None
red_avg_data = None
DO_SKY_CAL = False
MODEL_POL_CONVENTION = None
MODEL_VIS_UNITS = None

### Load classifications that use diffs if diffs are not available

In [ ]:
if not USE_DIFF:
    def read_antenna_classification(df, category):
        ac = ant_class.AntennaClassification()
        ac._data = {}
        for antname, class_data, antclass in zip(df['Antenna'], df[category], df[f'{category} Class']):
            try:        
                class_data = float(class_data)
            except:
                pass
            if isinstance(class_data, str) or np.isfinite(class_data):
                ant = (int(antname[:-1]), utils._comply_antpol(antname[-1]))
                ac[ant] = antclass
                ac._data[ant] = class_data
        return ac

    df = pd.read_csv(RTP_ANTCLASS)
    am_totally_dead = read_antenna_classification(df, 'Dead?')
    am_corr = read_antenna_classification(df, 'Low Correlation')
    am_xpol = read_antenna_classification(df, 'Cross-Polarized')
    zeros_class = read_antenna_classification(df, 'Even/Odd Zeros')
    xengine_diff_class = read_antenna_classification(df, 'Bad Diff X-Engines')

### Run `ant_metrics`

This classifies antennas as cross-polarized, low-correlation, or dead. Such antennas are excluded from any calibration.

In [ ]:
if USE_DIFF:
    am = ant_metrics.AntennaMetrics(SUM_FILE, DIFF_FILE, sum_data=data, diff_data=diff_data)
    am.iterative_antenna_metrics_and_flagging(crossCut=am_xpol_bad[1], deadCut=am_corr_bad[1])
    am.all_metrics = {}  # this saves time and disk by getting rid of per-iteration information we never use
    if SAVE_RESULTS:
        am.save_antenna_metrics(AM_FILE, overwrite=True)

In [ ]:
if USE_DIFF:
    # Turn ant metrics into classifications
    totally_dead_ants = [ant for ant, i in am.xants.items() if i == -1]
    am_totally_dead = ant_class.AntennaClassification(good=[ant for ant in ants if ant not in totally_dead_ants], bad=totally_dead_ants)
    am_corr = ant_class.antenna_bounds_checker(am.final_metrics['corr'], bad=[am_corr_bad], suspect=[am_corr_suspect], good=[(0, 1)])
    am_xpol = ant_class.antenna_bounds_checker(am.final_metrics['corrXPol'], bad=[am_xpol_bad], suspect=[am_xpol_suspect], good=[(-1, 1)])
ant_metrics_class = am_totally_dead + am_corr + am_xpol
if np.all([ant_metrics_class[utils.split_bl(bl)[0]] == 'bad' for bl in auto_bls]):
    ALL_FLAGGED = True
    print('All antennas are flagged for ant_metrics.')

### Mark sun-up data as bad and near-sunrise/sunset data as suspect

In [ ]:
max_sun_alt = np.max(utils.get_sun_alt(hd.times))
solar_class = ant_class.antenna_bounds_checker({ant: max_sun_alt for ant in ants}, good=[good_solar_altitude],
                                               suspect=[suspect_solar_altitude], bad=[bad_solar_altitude])


### Classify antennas responsible for 0s in visibilities as bad: 

This classifier looks for X-engine failure or packet loss specific to an antenna which causes either the even visibilities (or the odd ones, or both) to be 0s.

In [ ]:
if USE_DIFF:
    zeros_class = ant_class.even_odd_zeros_checker(data, diff_data, good=good_zeros_per_eo_spectrum, suspect=suspect_zeros_per_eo_spectrum)
if np.all([zeros_class[utils.split_bl(bl)[0]] == 'bad' for bl in auto_bls]):
    ALL_FLAGGED = True
    print('All antennas are flagged for too many even/odd zeros.')

### Examine and classify autocorrelation power and slope

These classifiers look for antennas with too high or low power or to steep a slope.

In [ ]:
auto_power_class = ant_class.auto_power_checker(data, good=auto_power_good, suspect=auto_power_suspect)
auto_slope_class = ant_class.auto_slope_checker(data, good=auto_slope_good, suspect=auto_slope_suspect, edge_cut=100, filt_size=17)
if np.all([(auto_power_class + auto_slope_class)[utils.split_bl(bl)[0]] == 'bad' for bl in auto_bls]):
    ALL_FLAGGED = True
    print('All antennas are flagged for bad autocorrelation power/slope.')
overall_class = auto_power_class + auto_slope_class + zeros_class + ant_metrics_class + solar_class

### Find starting set of array flags

In [ ]:
if not all_flagged():
    antenna_flags, array_flags = xrfi.flag_autos(data, flag_method="channel_diff_flagger", nsig=RFI_NSIG * 5, 
                                                 antenna_class=overall_class, flag_broadcast_thresh=.5)
    for key in antenna_flags:
        antenna_flags[key] = array_flags
    cache = {}
    _, array_flags = xrfi.flag_autos(data, freqs=data.freqs, flag_method="dpss_flagger",
                                     nsig=RFI_NSIG, antenna_class=overall_class,
                                     filter_centers=[0], filter_half_widths=[RFI_DPSS_HALFWIDTH],
                                     eigenval_cutoff=[1e-9], flags=antenna_flags, mode='dpss_matrix', 
                                     cache=cache, flag_broadcast_thresh=.5)

### Classify antennas based on non-noiselike diffs

In [ ]:
if not all_flagged():
    if USE_DIFF:
        xengine_diff_class = ant_class.non_noiselike_diff_by_xengine_checker(data, diff_data, flag_waterfall=array_flags, 
                                                                             antenna_class=overall_class, 
                                                                             xengine_chans=96, bad_xengine_zcut=bad_xengine_zcut)
        
        if np.all([overall_class[utils.split_bl(bl)[0]] == 'bad' for bl in auto_bls]):
            ALL_FLAGGED = True
            print('All antennas are flagged after flagging non-noiselike diffs.')
    overall_class += xengine_diff_class

### Examine and classify autocorrelation excess RFI and shape, finding consensus RFI mask along the way

This classifier iteratively identifies antennas for excess RFI (characterized by RMS of DPSS-filtered autocorrelations after RFI flagging) and bad shape, as determined by a discrepancy with the mean good normalized autocorrelation's shape. Along the way, it iteratively discovers a conensus array-wide RFI mask.

In [ ]:
def auto_bl_zscores(data, flag_array, cache={}):
    '''This function computes z-score arrays for each delay-filtered autocorrelation, normalized by the expected noise. 
    Flagged times/channels for the whole array are given 0 weight in filtering and are np.nan in the z-score.'''
    zscores = {}
    for bl in auto_bls:
        wgts = np.array(np.logical_not(flag_array), dtype=np.float64)
        model, _, _ = dspec.fourier_filter(hd.freqs, data[bl], wgts, filter_centers=[0], filter_half_widths=[RFI_DPSS_HALFWIDTH], mode='dpss_solve',
                                            suppression_factors=[1e-9], eigenval_cutoff=[1e-9], cache=cache)
        res = data[bl] - model
        int_time = 24 * 3600 * np.median(np.diff(data.times))
        chan_res = np.median(np.diff(data.freqs))
        int_count = int(int_time * chan_res)
        sigma = np.abs(model) / np.sqrt(int_count / 2)
        zscores[bl] = res / sigma    
        zscores[bl][flag_array] = np.nan

    return zscores

In [ ]:
def rfi_from_avg_autos(data, auto_bls_to_use, prior_flags=None, nsig=RFI_NSIG):
    '''Average together all baselines in auto_bls_to_use, then find an RFI mask by looking for outliers after DPSS filtering.'''
    
    # If there are no good autos, return 100% flagged
    if len(auto_bls_to_use) == 0:
        return np.ones(data[next(iter(data))].shape, dtype=bool)
    
    # Compute int_count for all unflagged autocorrelations averaged together
    int_time = 24 * 3600 * np.median(np.diff(data.times_by_bl[auto_bls[0][0:2]]))
    chan_res = np.median(np.diff(data.freqs))
    int_count = int(int_time * chan_res) * len(auto_bls_to_use)
    avg_auto = {(-1, -1, 'ee'): np.mean([data[bl] for bl in auto_bls_to_use], axis=0)}
    
    # Flag RFI first with channel differences and then with DPSS
    antenna_flags, _ = xrfi.flag_autos(avg_auto, int_count=int_count, nsig=(nsig * 5))
    if prior_flags is not None:
        antenna_flags[(-1, -1, 'ee')] = prior_flags
    _, rfi_flags = xrfi.flag_autos(avg_auto, int_count=int_count, flag_method='dpss_flagger',
                                   flags=antenna_flags, freqs=data.freqs, filter_centers=[0],
                                   filter_half_widths=[RFI_DPSS_HALFWIDTH], eigenval_cutoff=[1e-9], nsig=nsig)

    return rfi_flags


def flag_mostly_flagged_bands(flags):
    '''If at least MAX_BAND_FLAG_FRAC of a band (below or above FM) is flagged at some
    integration, flag that whole band there in place.'''
    for label, band in [('below FM', hd.freqs < FM_LOW_FREQ * 1e6), ('above FM', hd.freqs > FM_HIGH_FREQ * 1e6)]:
        for tind in range(flags.shape[0]):
            if not np.all(flags[tind, band]) and np.mean(flags[tind, band]) >= MAX_BAND_FLAG_FRAC:
                flags[tind, band] = True
                print(f'The band {label} is over {MAX_BAND_FLAG_FRAC:.0%} flagged at integration {tind}; '
                      'flagging it entirely.')

In [ ]:
# Iteratively develop RFI mask, excess RFI classification, and autocorrelation shape classification
if not all_flagged():
    stage = 1
    rfi_flags = np.array(array_flags)
    prior_end_states = set()
    while True:
        # compute DPSS-filtered z-scores with current array-wide RFI mask
        zscores = auto_bl_zscores(data, rfi_flags)
        rms = {bl: np.nanmean(zscores[bl]**2)**.5 if np.any(np.isfinite(zscores[bl])) else np.inf for bl in zscores}
        
        # figure out which autos to use for finding new set of flags
        candidate_autos = [bl for bl in auto_bls if overall_class[utils.split_bl(bl)[0]] != 'bad']
        if stage == 1:
            # use best half of the unflagged antennas
            med_rms = np.nanmedian([rms[bl] for bl in candidate_autos])
            autos_to_use = [bl for bl in candidate_autos if rms[bl] <= med_rms]
        elif stage == 2:
            # use all unflagged antennas which are auto RFI good, or the best half, whichever is larger
            med_rms = np.nanmedian([rms[bl] for bl in candidate_autos])
            best_half_autos = [bl for bl in candidate_autos if rms[bl] <= med_rms]
            good_autos = [bl for bl in candidate_autos if (overall_class[utils.split_bl(bl)[0]] != 'bad')
                          and (auto_rfi_class[utils.split_bl(bl)[0]] == 'good')]
            autos_to_use = (best_half_autos if len(best_half_autos) > len(good_autos) else good_autos)
        elif stage == 3:
            # use all unflagged antennas which are auto RFI good or suspect
            autos_to_use = [bl for bl in candidate_autos if (overall_class[utils.split_bl(bl)[0]] != 'bad')]
    
        # compute new RFI flags
        rfi_flags = rfi_from_avg_autos(data, autos_to_use)
    
        # perform auto shape and RFI classification
        overall_class = auto_power_class + auto_slope_class + zeros_class + ant_metrics_class + solar_class + xengine_diff_class
        auto_rfi_class = ant_class.antenna_bounds_checker(rms, good=auto_rfi_good, suspect=auto_rfi_suspect, bad=(0, np.inf))
        overall_class += auto_rfi_class
        auto_shape_class = ant_class.auto_shape_checker(data, good=auto_shape_good, suspect=auto_shape_suspect,
                                                        flag_spectrum=np.sum(rfi_flags, axis=0).astype(bool), 
                                                        antenna_class=overall_class)
        overall_class += auto_shape_class
        
        # check for convergence by seeing whether we've previously gotten to this number of flagged antennas and channels
        if stage == 3:
            if (len(overall_class.bad_ants), np.sum(rfi_flags)) in prior_end_states:
                break
            prior_end_states.add((len(overall_class.bad_ants), np.sum(rfi_flags)))
        else:
            stage += 1

    flag_mostly_flagged_bands(rfi_flags)


In [ ]:
auto_class = auto_power_class + auto_slope_class
if auto_rfi_class is not None:
    auto_class += auto_rfi_class
if auto_shape_class is not None:
    auto_class += auto_shape_class
if np.all([overall_class[utils.split_bl(bl)[0]] == 'bad' for bl in auto_bls]):
    ALL_FLAGGED = True
    print('All antennas are flagged after flagging for bad autos power/slope/rfi/shape.')

In [ ]:
if not all_flagged():
    def rfi_plot(cls, flags=rfi_flags):
        avg_auto = {(-1, -1, 'ee'): np.mean([data[bl] for bl in auto_bls if not cls[utils.split_bl(bl)[0]] == 'bad'], axis=0)}
        plt.figure(figsize=(12, 5), dpi=100)
        plt.semilogy(hd.freqs / 1e6, np.where(flags, np.nan, avg_auto[(-1, -1, 'ee')])[0], label = 'Average Good or Suspect Autocorrelation', zorder=100)
        plt.semilogy(hd.freqs / 1e6, np.where(False, np.nan, avg_auto[(-1, -1, 'ee')])[0], 'r', lw=.5, label=f'{np.sum(flags[0])} Channels Flagged for RFI')
        plt.legend()
        plt.xlabel('Frequency (MHz)')
        plt.ylabel('Uncalibrated Autocorrelation')
        plt.tight_layout()

# *Figure 1: RFI Flagging*

This figure shows RFI identified using the average of all autocorrelations---excluding bad antennas---for the first integration in the file.

In [ ]:
if not all_flagged(): rfi_plot(overall_class)

In [ ]:
def autocorr_plot(cls):    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100, sharey=True, gridspec_kw={'wspace': 0})
    labels = []
    colors = ['darkgreen', 'goldenrod', 'maroon']
    for ax, pol in zip(axes, antpols):
        for ant in cls.ants:
            if ant[1] == pol:
                color = colors[cls.quality_classes.index(cls[ant])]
                ax.semilogy(hd.freqs / 1e6, np.mean(data[utils.join_bl(ant, ant)], axis=0), color=color, lw=.5)
        ax.set_xlabel('Frequency (MHz)', fontsize=12)
        ax.set_title(f'{utils.join_pol(pol, pol)}-Polarized Autos')

    axes[0].set_ylabel('Raw Autocorrelation', fontsize=12)
    axes[1].legend([matplotlib.lines.Line2D([0], [0], color=color) for color in colors], 
                   [cl.capitalize() for cl in cls.quality_classes], ncol=1, fontsize=12, loc='upper right', framealpha=1)
    plt.tight_layout()

# *Figure 2: Plot of autocorrelations with classifications*
This figure shows a plot of all autocorrelations in the array, split by polarization. 
Antennas are classified based on their autocorrelations into good, suspect, and bad, by examining power, slope, and RFI-occupancy.

In [ ]:
if not all_flagged(): autocorr_plot(auto_class)

### Summarize antenna classification prior to calibration

In [ ]:
def array_class_plot(cls, extra_label=""):
    outriggers = [ant for ant in hd.data_ants if ant >= 320]

    if len(outriggers) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100, gridspec_kw={'width_ratios': [2, 1]})
        plot_antclass(hd.antpos, cls, ax=axes[0], ants=[ant for ant in hd.data_ants if ant < 320], legend=False, title=f'HERA Core{extra_label}')
        plot_antclass(hd.antpos, cls, ax=axes[1], ants=outriggers, radius=50, title='Outriggers')
    else:
        fig, axes = plt.subplots(1, 1, figsize=(9, 6), dpi=100)
        plot_antclass(hd.antpos, cls, ax=axes, ants=[ant for ant in hd.data_ants if ant < 320], legend=False, title=f'HERA Core{extra_label}')

# *Figure 3: Summary of antenna classifications prior to calibration*
This figure shows the location and classification of all antennas prior to calibration. 
Antennas are split along the diagonal, with ee-polarized antpols represented by the southeast half of each antenna and nn-polarized antpols represented by the northwest half.
Outriggers are split from the core and shown at exaggerated size in the right-hand panel. This classification includes `ant_metrics`, a count of the zeros in the even or odd visibilities, and autocorrelation power, slope, and RFI occupancy.
An antenna classified as bad in *any* classification will be considered bad. 
An antenna marked as suspect *any* in any classification will be considered suspect unless it is also classified as bad elsewhere.

In [ ]:
if not all_flagged(): array_class_plot(overall_class)

In [ ]:
# delete diffs to save memory
if USE_DIFF:
    del diff_data, hd_diff
try:
    del cache
except NameError:
    pass
malloc_trim()

## Load and match the sky model

The sky model is expected to be LST-stacked, redundantly-averaged, filtered visibilities from prior nights (e.g. `zen.*.sum.sky_model.uvh5` corner-turn products), including autocorrelations. Model files are matched to this file's LSTs, loaded at the matched times, rephased to the exact data LSTs, and wrapped in a `RedDataContainer` so that any data baseline indexes into its redundant group's model. If no matching model can be found, calibration is skipped entirely and placeholder outputs are written.

In [ ]:
if not all_flagged():
    if SKY_MODEL_FILES_GLOB is not None:
        sky_model_files = sorted(glob.glob(SKY_MODEL_FILES_GLOB))
    else:
        # try to find files at NRAO
        sky_model_files = sorted(glob.glob('/lustre/aoc/projects/hera/h6c-analysis/abscal_models/'
                                           'h6c_filtered_lst_stack/zen.*.sum.sky_model.uvh5'))
    print(f'Found {len(sky_model_files)} sky model files'
          + (' in ' + os.path.dirname(sky_model_files[0]) if len(sky_model_files) > 0 else '') + '.')

    if len(sky_model_files) == 0:
        print('No sky model files found... skipping sky calibration and writing placeholder outputs.')
    else:
        model_match_start = time.time()
        matched_model_files = sorted(set(abscal.match_times(SUM_FILE, sky_model_files, filetype='uvh5')))
        if len(matched_model_files) == 0:
            print(f'No sky model files found matching the LSTs of this file after searching for '
                  f'{(time.time() - model_match_start) / 60:.2f} minutes. Skipping sky calibration.')
        else:
            DO_SKY_CAL = True
            hdm = io.HERAData(matched_model_files)
            all_model_times, all_model_lsts = abscal.get_all_times_and_lsts(hdm, unwrap=True)
            d2m_time_map = abscal.get_d2m_time_map(data.times, np.unwrap(data.lsts),
                                                   all_model_times, all_model_lsts, extrap_limit=.5)

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    # load matched times for all polarizations the model has, then rephase to the exact data LSTs
    model_pols = hdm.pols
    if isinstance(model_pols, dict):  # multiple model files
        model_pols = sorted(set(pol for pols in model_pols.values() for pol in pols))
    pols_to_load = [pol for pol in ['ee', 'nn', 'en', 'ne'] if pol in model_pols]
    model, model_flags, _ = io.partial_time_io(hdm, np.unique([d2m_time_map[time] for time in data.times]),
                                               polarizations=pols_to_load)
    model_blvecs = {bl: model.antpos[bl[0]] - model.antpos[bl[1]] for bl in model.keys()}
    utils.lst_rephase(model, model_blvecs, model.freqs, data.lsts - model.lsts,
                      lat=hdm.telescope.location.lat.deg, inplace=True)

    # record model metadata for the output calfits
    def _scalar(attr):
        return next(iter(attr.values())) if isinstance(attr, dict) else attr
    MODEL_POL_CONVENTION = _scalar(getattr(hdm, 'pol_convention', None))
    MODEL_VIS_UNITS = _scalar(getattr(hdm, 'vis_units', None))

    # wrap in RedDataContainers keyed by redundant group, using the FULL antenna
    # table (model baselines can involve antennas not in this file)
    all_reds = redcal.get_reds(hd.antpos, pols=pols_to_load)
    model = datacontainer.RedDataContainer(model, reds=all_reds)
    model_flags = datacontainer.RedDataContainer(model_flags, reds=all_reds)
    print(f'Loaded and rephased sky model for pols {pols_to_load} at '
          f'{len(np.unique([d2m_time_map[time] for time in data.times]))} matched times.')

## Antenna → SNAP mapping from M&C

Antenna → SNAP assignments are queried from the CM database at this file's JD via `hera_notebook_templates.utils.get_ant_to_snap_dict` (the walk HH → A → FDV → FEM → NBP → PAM → SNP, with `part_rosetta` supplying hostnames; hard error if M&C is unreachable). Decoherence states are per-SNAP, so this mapping is what ties antennas to shared X-engine packet-loss states.

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    from hera_notebook_templates import utils as hnbt_utils
    ant_snap = hnbt_utils.get_ant_to_snap_dict(np.mean(data.times), sorted({ant[0] for ant in ants}))
    snap_node_slot, snap_sort_key, snap_label = hnbt_utils.snap_labelers(ant_snap.values())
    print(f'{len(ant_snap)} antennas mapped to {len(set(ant_snap.values()))} SNAPs.')

## Antenna identity and coherence audit

Before any calibration, `hera_qm.ant_class.antenna_identity_checker` checks every antenna's visibilities against their own redundant group's sky model with a delay-searched phase coherence statistic (needing only raw data, the model, and the RFI mask). A correctly-labeled, working antenna's visibilities are highly coherent with their own model after fitting a single delay; visibilities carrying another antenna's signal (e.g. from cabling errors that cyclically permute antennas within a node) are incoherent with their own model but highly coherent with the true antenna's.

Low-coherence antennas are scanned against all node-mates. Decisive identifications (bounds above) are repaired by relabeling — a pure bookkeeping fix, no data values change — if `AUTO_REPAIR_IDENTITIES`; otherwise they are classified as bad. Repairs are recorded in Table 1, the ant_class CSV, and the output files' histories. Note that the SNAP assignment belongs to the correlator *input*, so a relabeled antenna inherits the SNAP its visibilities came through, keeping decoherence bookkeeping correct. Antennas with low coherence but no decisive alternative identity are classified as suspect or bad by the same bounds — so beyond mislabelings, the audit also detects antennas likely to be highly discrepant with the sky model (e.g. broken or cross-polarized ones, which are flagged, not repaired).

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    audit_start = time.time()
    # baselines for the audit: modeled co-pol cross-correlations between not-bad, SNAP-mapped antennas
    audit_bls = []
    for bl in data:
        if bl[0] == bl[1] or bl[2] not in ('ee', 'nn'):
            continue
        if SKIP_OUTRIGGERS and (bl[0] >= 320 or bl[1] >= 320):
            continue
        if (bl[0] not in ant_snap) or (bl[1] not in ant_snap):
            continue
        if any(ant in overall_class.bad_ants for ant in utils.split_bl(bl)):
            continue
        if not (SKYCAL_MIN_BL_LEN <= np.linalg.norm(hd.antpos[bl[1]] - hd.antpos[bl[0]]) <= SKYCAL_MAX_BL_LEN):
            continue
        if bl in model:
            audit_bls.append(bl)

    # candidate identities for a mislabeled antenna: everyone in the same node
    node_of = {antnum: snap_node_slot(snap)[0] for antnum, snap in ant_snap.items()}
    candidate_groups = {antnum: [a for a in node_of if node_of[a] == node_of[antnum]]
                        for antnum in node_of}
    identity_class, labeled_to_true, self_coherence = ant_class.antenna_identity_checker(
        data, model, audit_bls, candidate_groups, good=identity_coherence_good,
        suspect=identity_coherence_suspect, repair_margin=identity_repair_margin,
        flag_waterfall=rfi_flags)
    print(f'Finished identity audit in {(time.time() - audit_start) / 60:.2f} minutes.')

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    if AUTO_REPAIR_IDENTITIES and len(labeled_to_true) > 0:
        # relabel the visibilities: each mislabeled antenna's data gets its true antenna
        # number. No data values change -- this is bookkeeping repair of a cabling/M&C error.
        def fix_key(bl):
            return (labeled_to_true.get(bl[0], bl[0]), labeled_to_true.get(bl[1], bl[1]), bl[2])
        data = datacontainer.DataContainer({fix_key(bl): data[bl] for bl in data})
        data.freqs, data.times, data.lsts = hd.freqs, hd.times, hd.lsts

        # the SNAP assignment belongs to the correlator input: we thought one antenna was
        # plugged into a given SNAP input but it was actually another, so the true antenna
        # inherits the SNAP of the input its visibilities were labeled with
        repaired_ant_snap = dict(ant_snap)
        for labeled, true_ant in labeled_to_true.items():
            repaired_ant_snap[true_ant] = ant_snap[labeled]
        ant_snap = repaired_ant_snap

        # antenna-keyed classifications describe the visibilities, so they all travel with the
        # relabeling: remap every AntennaClassification in the notebook's namespace
        def remap_ant(ant):
            return (labeled_to_true.get(ant[0], ant[0]), ant[1])
        def remap_classification(ac):
            remapped = ant_class.AntennaClassification()
            remapped._data = {}
            for ant in ac.ants:
                remapped[remap_ant(ant)] = ac[ant]
                if ant in getattr(ac, '_data', {}):
                    remapped._data[remap_ant(ant)] = ac._data[ant]
            return remapped
        for name, cls in list(globals().items()):
            if isinstance(cls, ant_class.AntennaClassification):
                globals()[name] = remap_classification(cls)
        ants = sorted({remap_ant(ant) for ant in ants})
        auto_bls = [bl for bl in data if (bl[0] == bl[1]) and (utils.split_pol(bl[2])[0] == utils.split_pol(bl[2])[1])]

        identity_repair_note = ('Identity repairs applied: '
                                + '; '.join(f'visibilities labeled {labeled} reassigned to antenna {true_ant}'
                                            for labeled, true_ant in sorted(labeled_to_true.items())) + '.')
        print(identity_repair_note)
    elif len(labeled_to_true) > 0:
        print('AUTO_REPAIR_IDENTITIES is False: repairs identified above are NOT applied; '
              'affected antennas are classified as bad instead.')
        for ant in list(identity_class.ants):
            if ant[0] in labeled_to_true:
                identity_class[ant] = 'bad'
    overall_class += identity_class

In [ ]:
def node_identity_plot():
    core = sorted(antnum for antnum in hd.data_ants if antnum < 320 and antnum in ant_snap)
    nodes = sorted({node_of[antnum] for antnum in core})
    cmap = plt.get_cmap('tab20')
    fig, ax = plt.subplots(1, 1, figsize=(9, 7), dpi=100)
    for k, node in enumerate(nodes):
        antnums = [antnum for antnum in core if node_of[antnum] == node]
        xpos = [hd.antpos[antnum][0] for antnum in antnums]
        ypos = [hd.antpos[antnum][1] for antnum in antnums]
        ax.scatter(xpos, ypos, s=300, color=cmap(k % 20), alpha=.5, edgecolors='k', lw=.5)
        for antnum in antnums:
            ax.text(hd.antpos[antnum][0], hd.antpos[antnum][1], str(antnum),
                    ha='center', va='center', fontsize=7)
        ax.text(np.median(xpos), np.median(ypos) + 8, f'N{node:02d}', fontsize=12, fontweight='bold',
                ha='center', va='center', color=cmap(k % 20),
                bbox=dict(facecolor='w', alpha=.7, edgecolor='none', pad=1))
    if len(labeled_to_true) > 0:
        for labeled, true_ant in labeled_to_true.items():
            ax.annotate('', xy=hd.antpos[true_ant][:2], xytext=hd.antpos[labeled][:2],
                        arrowprops=dict(arrowstyle='-|>', color='r', lw=2.5, shrinkA=10, shrinkB=10))
        arrow_handle = matplotlib.lines.Line2D([0], [0], color='r', lw=2.5, marker='>',
                                               markevery=(1, 1), markersize=8)
        ax.legend(handles=[arrow_handle], labels=['Labeled As --> True Identity'], loc='upper right')
    ax.axis('equal')
    ax.set_xlabel('East-West Position (meters)')
    ax.set_ylabel('North-South Position (meters)')
    ax.set_title('Antennas by Node')
    plt.tight_layout()

# *Figure 4: Array nodes and antenna identities*

Non-outrigger antennas colored by their node (labeled at each node's median position). If the identity audit found mislabeled antennas, red arrows point from the antenna a set of visibilities was *labeled* as to the antenna the audit determined they actually belong to — cabling permutations appear as cycles of arrows within a node.

In [ ]:
if not all_flagged() and DO_SKY_CAL: node_identity_plot()

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    # select the cross-correlations to calibrate with: modeled, length-limited,
    # both antennas mapped to SNAPs, neither antenna bad
    bls = []
    unmapped = set()
    for bl in data:
        if bl[0] == bl[1] or bl[2] not in ('ee', 'nn'):
            continue
        if SKIP_OUTRIGGERS and (bl[0] >= 320 or bl[1] >= 320):
            continue
        if (bl[0] not in ant_snap) or (bl[1] not in ant_snap):
            unmapped |= {a for a in bl[:2] if a not in ant_snap}
            continue
        if any(ant in overall_class.bad_ants for ant in utils.split_bl(bl)):
            continue
        if not (SKYCAL_MIN_BL_LEN <= np.linalg.norm(hd.antpos[bl[1]] - hd.antpos[bl[0]]) <= SKYCAL_MAX_BL_LEN):
            continue
        if bl not in model:
            continue
        bls.append(bl)
    if len(unmapped) > 0:
        print(f'Excluding antennas without M&C SNAP mappings: {sorted(unmapped)}')

    # the per-channel refinement requires uniform flagging across baselines, so keep only the
    # groups sharing each polarization's most common exact model flag/hole pattern (their shared
    # flags simply become channel flags); a group with extra holes (e.g. from partial LST
    # coverage in the stacked model) would otherwise flag a strict subset of baselines
    pattern = {bl: (model_flags[bl] | ~np.isfinite(model[bl]) | (model[bl] == 0)).tobytes() for bl in bls}
    for pol in sorted({bl[2] for bl in bls}):
        patterns_here = [pattern[bl] for bl in bls if bl[2] == pol]
        modal = max(set(patterns_here), key=patterns_here.count)
        if patterns_here.count(modal) < len(patterns_here):
            print(f'Dropping {len(patterns_here) - patterns_here.count(modal)} of {len(patterns_here)} '
                  f'{pol} baselines whose model flag pattern differs from the most common one.')
            bls = [bl for bl in bls if bl[2] != pol or pattern[bl] == modal]
    n_ee = sum(bl[2] == 'ee' for bl in bls)
    n_nn = sum(bl[2] == 'nn' for bl in bls)
    print(f'{len(bls)} modeled cross-correlations selected for calibration ({n_ee} ee, {n_nn} nn).')

    # sky_calibrate expects a dict of per-antenna flag waterfalls, but its per-channel
    # refinement requires uniform flagging across antennas, so every antenna gets the same
    # array-wide RFI flags (by reference, not copies)
    per_ant_rfi_flags = {ant: rfi_flags for bl in bls for ant in utils.split_bl(bl)}

    # pass only the co-polarized autocorrelations: cross-polarized "autos" are
    # not autocorrelation power and would corrupt the starting amplitudes.
    # dt and df are stated explicitly since this container has no metadata.
    autos = datacontainer.DataContainer({bl: data[bl] for bl in auto_bls})
    int_time = 24 * 3600 * np.median(np.diff(data.times))
    chan_res = np.median(np.diff(data.freqs))

## Perform iterative sky-model calibration

`skycal.sky_calibrate` runs the full staged chain — data/model ratio, model-based firstcal delays and offsets, autocorrelation-referenced starting amplitudes, then per-channel Gauss-Newton refinement restricted to inter-SNAP cross-correlations (so per-SNAP signal loss lands cleanly in the refined gains). In place of `redcal`'s chi^2 per antenna, antennas are classified by their normalized chi^2 against the sky model: the weighted mean over each antenna's baselines of $w_{ij} |Z_{ij} - g_i g_j^*|^2$, whose expectation is ~1 for noise-like residuals plus a model-error floor common to all antennas.

While antennas are still being excluded, individual channels are allowed to fail to converge (their gains come back `np.nan`); the median over channels used to rank antennas simply skips them, and the antenna that made them fail is usually the next one removed. Once the median finds no more offenders, any channel still failing to converge is a hard error, so the accepted solution has always converged everywhere. The exclusion loop mirrors `file_calibration`'s: absolute good/suspect chi^2 bounds with a dynamic-range rescue (only the worst tier of offenders is flagged each round, since a bad antenna inflates its partners' chi^2), a median-then-mean classification ladder on each solve, and a whole-polarization stop. The loop converges when neither metric finds new offenders — since the metric only changes classification, not the solution, no final re-solve is needed. One caveat specific to sky calibration: an antenna on a severely decoherent SNAP genuinely mismatches the model in the affected X-engine blocks, which inflates its chi^2 — the median over frequency deliberately dampens this, since suppression hits a minority of blocks.

In [ ]:
def sky_chisq(gains, meta):
    '''Computes normalized chi^2 waterfalls per antpol and per polarization from sky_calibrate outputs.
    For each antenna, chi^2 is the mean over its baselines of wgts * |data_model_ratio - g_i conj(g_j)|^2,
    which has expectation ~1 for noise-like residuals. Returns (cspa, total_chisq).'''
    num, den = {}, {}
    tot_num, tot_den = {}, {}
    for bl in meta['data_model_ratio']:
        ant_i, ant_j = utils.split_bl(bl)
        if ant_i not in gains or ant_j not in gains:
            continue
        z2 = meta['wgts'][bl] * np.abs(meta['data_model_ratio'][bl] - gains[ant_i] * np.conj(gains[ant_j]))**2
        ok = (meta['wgts'][bl] > 0) & np.isfinite(z2)
        z2 = np.where(ok, z2, 0)
        for ant in (ant_i, ant_j):
            num[ant] = num.get(ant, 0) + z2
            den[ant] = den.get(ant, 0) + ok
        jpol = ant_i[1]
        tot_num[jpol] = tot_num.get(jpol, 0) + z2
        tot_den[jpol] = tot_den.get(jpol, 0) + ok
    cspa = {ant: np.where(den[ant] > 0, num[ant] / np.where(den[ant] > 0, den[ant], 1), np.nan) for ant in num}
    total_chisq = {jpol: np.where(tot_den[jpol] > 0, tot_num[jpol] / np.where(tot_den[jpol] > 0, tot_den[jpol], 1), np.nan)
                   for jpol in tot_num}
    return cspa, total_chisq

def check_if_whole_pol_flagged(cls, pols=['Jee', 'Jnn'], thresh=.75):
    '''Checks if nearly an entire polarization is flagged (depending on thresh), including antennas
    already bad in overall_class. If it is, returns True and marks all antennas as bad in cls.'''
    combined = overall_class + cls
    flag_fracs = np.array([np.mean([combined[ant] == 'bad' for ant in combined.ants if ant[1] == pol]) for pol in pols])
    if np.any(flag_fracs > thresh):
        for pol, frac in zip(pols, flag_fracs):
            if frac > thresh:
                print(f'Polarization {pol} is {frac:.3%} flagged > {thresh:.3%} threshold. Stopping sky calibration.')
        for ant in cls:
            cls[ant] = 'bad'
        return True
    return False

def flag_high_chisq_antennas(cspa, metric):
    '''Classifies antennas by the median or mean (per metric) over RFI-unflagged cells of their
    chi^2 waterfalls, using absolute good/suspect bounds. Bad antennas much better than the worst
    are demoted to suspect, so only the worst tier is flagged per call (a bad antenna inflates its
    partners' chi^2, which should be re-evaluated after a re-solve without it). Prints any newly
    bad antennas and returns the classification.'''
    avg_alg = (np.nanmedian if metric == 'median' else np.nanmean)
    avg_cspa = {ant: avg_alg(np.where(rfi_flags, np.nan, cspa[ant])) for ant in cspa}
    cspa_class = ant_class.antenna_bounds_checker(avg_cspa, good=sc_cspa_good, suspect=sc_cspa_suspect,
                                                  bad=[(-np.inf, np.inf)])
    for ant in cspa_class.bad_ants:
        if avg_cspa[ant] < np.nanmax(list(avg_cspa.values())) / SC_MAX_CHISQ_FLAGGING_DYNAMIC_RANGE:
            cspa_class[ant] = 'suspect'
    if len(cspa_class.bad_ants) > 0:
        print(f'Removing {cspa_class.bad_ants} for high {metric} unflagged chi^2.')
        for ant in cspa_class.bad_ants:
            print(f'\t{ant}: {avg_cspa[ant]:.3f}')
    return cspa_class

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    skycal_start = time.time()
    skycal_class = ant_class.AntennaClassification(good=[ant for ant in ants if ant not in overall_class.bad_ants])

    # iteratively rerun sky calibration
    for i in range(SC_MAX_RERUN + 1):
        # check to see whether we're done because an entire pol is flagged
        if check_if_whole_pol_flagged(skycal_class):
            break

        cal_bls = [bl for bl in bls
                   if not any(ant in (overall_class + skycal_class).bad_ants for ant in utils.split_bl(bl))]
        # while antennas are still being removed, up to half the channels may fail: every round
        # re-solves every channel from scratch, so a failed channel gets a fresh chance after
        # each antenna removal (whatever antenna made it fail is usually the next one removed)
        gains, meta = skycal.sky_calibrate(data, model, autos=autos, model_flags=model_flags,
                                           ant_flags=per_ant_rfi_flags, ant_to_SNAP_dict=ant_snap, bls=cal_bls,
                                           dt=int_time, df=chan_res, refine_maxiter=SC_MAXITER,
                                           max_divergent_chan_frac=.5)
        malloc_trim()
        diverged = sorted({int(c) for chans in meta['divergent_chans'].values() for c in chans})
        cspa, total_chisq = sky_chisq(gains, meta)
        cspa_class = flag_high_chisq_antennas(cspa, 'median')
        if len(cspa_class.bad_ants) == 0:
            # the more sensitive mean on the same solution
            cspa_class = flag_high_chisq_antennas(cspa, 'mean')
        skycal_class += cspa_class

        # stop if a whole pol is now flagged, or if neither metric found new offenders
        if check_if_whole_pol_flagged(skycal_class) or len(cspa_class.bad_ants) == 0:
            break

    # channels still unconverged in the accepted solution cannot be calibrated with this data,
    # so up to SC_MAX_DIVERGENT_CHANS of them are flagged (both polarizations).
    if meta is not None and len(diverged) > 0:
        if len(diverged) > SC_MAX_DIVERGENT_CHANS:
            raise RuntimeError(
                f'{len(diverged)} channels failed to converge '
                f'({", ".join(f"{c} ({hd.freqs[c] / 1e6:.2f} MHz)" for c in diverged[:6])}'
                f'{", ..." if len(diverged) > 6 else ""}), exceeding SC_MAX_DIVERGENT_CHANS = '
                f'{SC_MAX_DIVERGENT_CHANS}. Do not proceed with unconverged gains.')
        for (tind, pol), chans in meta['divergent_chans'].items():
            if len(chans) > 0:
                rfi_flags[tind, np.asarray(chans, dtype=int)] = True
        print(f'Flagged {len(diverged)} channels that failed to converge: '
              f'{", ".join(f"{c} ({hd.freqs[c] / 1e6:.2f} MHz)" for c in diverged)}.')
        flag_mostly_flagged_bands(rfi_flags)

    if total_chisq is not None:
        for jpol in sorted(total_chisq):
            unflagged = np.where(rfi_flags, np.nan, total_chisq[jpol])
            print(f'{jpol} sky-model chi^2 (unflagged): mean = {np.nanmean(unflagged):.3f}, '
                  f'median = {np.nanmedian(unflagged):.3f}')
    print(f'Finished iterative sky calibration in {(time.time() - skycal_start) / 60:.2f} minutes.')
    overall_class += skycal_class

    # if everything ended up bad (e.g. a whole-polarization stop before the first
    # solve), later products cannot be built; fall through to placeholder outputs
    if np.all([ant in overall_class.bad_ants for ant in ants]):
        DO_SKY_CAL = False
        print('All antennas are now classified as bad. Skipping remaining sky calibration products.')


In [ ]:
def chisq_convergence_plot():
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), dpi=100, sharex=True, gridspec_kw={'hspace': .05})
    for jpol, color in zip(sorted(total_chisq), ['b', 'r']):
        for tind in range(len(data.times)):
            axes[0].plot(hd.freqs / 1e6, np.where(rfi_flags[tind], np.nan, total_chisq[jpol][tind]),
                         '.', ms=1, color=color, alpha=.5, label=(jpol if tind == 0 else None))
    axes[0].axhline(1, color='k', lw=.5)
    axes[0].set_yscale('log')
    axes[0].set_ylabel('Normalized $\\chi^2$')
    axes[0].legend(markerscale=20)

    for (tind, pol), iters in meta['iter'].items():
        axes[1].plot(hd.freqs / 1e6, np.where(rfi_flags[tind], np.nan, iters),
                     '.', ms=1, alpha=.5, label=f'JD {data.times[tind]:.6f} (t={tind}), {pol}')
    axes[1].set_ylabel('Refinement Iterations')
    axes[1].set_xlabel('Frequency (MHz)')
    axes[1].legend(markerscale=20)
    plt.tight_layout()

# *Figure 5: Sky calibration goodness-of-fit and convergence*
The top panel shows the total normalized $\chi^2$ against the sky model per polarization and integration; the expectation for noise-like residuals is 1, and a broadband floor above that reflects sky-model error. The bottom panel shows the number of Gauss-Newton refinement iterations each channel needed to converge; channels that stand out here are ill-conditioned (e.g. heavily flagged or low-signal).

In [ ]:
if not all_flagged() and DO_SKY_CAL: chisq_convergence_plot()

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    deco1_start = time.time()
    # initial decoherence estimate from the first calibration pass, used to correct the
    # redundant averaging below; it is re-estimated after the final calibration
    logamp_wgts = skycal.log_gain_inverse_variance(meta['wgts'], meta['g0'], meta['refined_gains'], ant_snap)
    logamp_wgts = {ant: np.where(rfi_flags, 0, wgt) for ant, wgt in logamp_wgts.items()}
    deco1, dmeta1 = skycal.estimate_SNAP_decoherence(gains, logamp_wgts, ant_snap, np.asarray(hd.freqs))
    sd1 = io.SNAPDecoherence.from_estimate(deco1, dmeta1, ant_snap, np.asarray(hd.freqs), np.asarray(hd.times))
    print(f'Finished initial decoherence estimation in {(time.time() - deco1_start) / 60:.2f} minutes.')

### Cross-based RFI flagging on redundantly-averaged visibilities

An initial decoherence estimate is made from the first calibration pass so that the redundant average can use **all** visibilities consistently: the gains are cleaned of the fitted per-SNAP staircase, and the exact baseline-class-aware correction (`apply_cal.correct_SNAP_decoherence_in_place`) is applied to inter-SNAP cross-correlations — without this, calibrating with suppression-carrying gains over-corrects intra-SNAP baselines and autos, smearing block structure into the averages. The corrected, calibrated data are then averaged with inverse-variance weights from the calibrated autocorrelations (`apply_cal.calibrate_and_red_avg`, whose returned effective nsamples make the noise prediction below exact), converted to SNR units, high-pass delay filtered (independently below and above the FM band, with per-channel leverage correction so residual z-scores are unbiased even at band edges), and incoherently averaged over all baseline groups with at least `CROSS_RFI_MIN_SAMP_FRAC` of the maximum number of samples. The resulting per-polarization z-scores dig out low-level RFI invisible to the autocorrelation-based mask; outliers and their watershed neighbors are flagged here (entering all downstream products through the flag mask and the decoherence fit weights — since every channel is calibrated independently, no re-solve is needed), and the z-scores are saved as a `UVFlag` metrics file for full-day synthesis across the night by `full_day_rfi_round_2`.

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    cross_rfi_start = time.time()
    # unfiltered reds (exclusion is handled by ex_ants) with autos, so every listed antenna's
    # auto key resolves to the averaged autocorrelation for noise prediction below
    red_avg_reds = redcal.get_reds(hd.antpos, pols=['ee', 'nn'], include_autos=True)
    red_avg_data, red_avg_flags, red_avg_nsamples, _ = apply_cal.calibrate_and_red_avg(
        data, gains, red_avg_reds, ant_flags={ant: rfi_flags for ant in gains},
        ex_ants=overall_class.bad_ants, snap_decoherence=sd1, dt=int_time, df=chan_res, compute_chisq=False)

    # delay filter the well-sampled, short-enough groups in SNR units: groups with at least
    # CROSS_RFI_MIN_SAMP_FRAC of the median nsamples of that polarization's averaged autocorrelation
    low_band = slice(0, np.argwhere(hd.freqs > FM_LOW_FREQ * 1e6)[0][0])
    high_band = slice(np.argwhere(hd.freqs > FM_HIGH_FREQ * 1e6)[0][0], len(hd.freqs))
    med_auto_nsamples = {bl[2]: np.median(red_avg_nsamples[bl]) for bl in red_avg_data if bl[0] == bl[1]}
    bls_to_filter = [bl for bl in red_avg_data if bl[0] != bl[1]
                     and np.median(red_avg_nsamples[bl]) >= med_auto_nsamples[bl[2]] * CROSS_RFI_MIN_SAMP_FRAC
                     and np.linalg.norm(hd.antpos[bl[0]] - hd.antpos[bl[1]]) / constants.c * 1e9 < CROSS_RFI_FILTER_DELAY]
    filter_wgts = (~np.all(list(red_avg_flags.values()), axis=0)).astype(float)

    # per-channel leverage of the DPSS filter: the residual of a linear fit has variance
    # sigma^2 (1 - h_k), so without this correction z-scores are biased low, worst at band
    # edges where leverage is largest
    leverage = np.ones_like(filter_wgts)
    for band in [low_band, high_band]:
        basis = np.asarray(dspec.dpss_operator(hd.freqs[band], [0], [CROSS_RFI_FILTER_DELAY / 1e9],
                                               eigenval_cutoff=[1e-12])[0]).real
        for tind in range(filter_wgts.shape[0]):
            wgt_here = filter_wgts[tind, band]
            basis_wgted = basis * wgt_here[:, None]
            with np.errstate(all='ignore'):
                mapping = np.linalg.pinv(basis.T @ basis_wgted) @ basis_wgted.T
            leverage[tind, band] = np.einsum('cm,mc->c', basis, mapping)
    resid_ok = (filter_wgts > 0) & (leverage < 0.9)
    resid_norm = np.sqrt(np.clip(1 - leverage, 0.1, None))

    cache = {}
    filter_kwargs = dict(filter_centers=[0], filter_half_widths=[CROSS_RFI_FILTER_DELAY / 1e9],
                         eigenval_cutoff=[1e-12], suppression_factors=[1e-12])
    dly_filt_SNRs = {}
    for bl in bls_to_filter:
        noise_var = noise.predict_noise_variance_from_autos(bl, red_avg_data, dt=int_time, df=chan_res,
                                                            nsamples=red_avg_nsamples)
        with np.errstate(all='ignore'):
            snr = red_avg_data[bl] / noise_var**.5
        snr_mdl = np.zeros_like(snr)
        for band in [low_band, high_band]:
            snr_mdl[:, band], _, _ = dspec.fourier_filter(hd.freqs[band], snr[:, band],
                                                          wgts=filter_wgts[:, band], mode='dpss_solve',
                                                          max_contiguous_edge_flags=len(hd.freqs),
                                                          cache=cache, **filter_kwargs)
        dly_filt_SNRs[bl] = np.where(resid_ok, (snr - snr_mdl) / resid_norm, np.nan)
    print(f'Delay-filtered {len(bls_to_filter)} redundantly-averaged baseline groups.')

    # incoherent average over baselines -> per-polarization z-scores. After the leverage
    # correction each residual is unit-variance complex, so |resid| is Rayleigh with
    # mean sqrt(pi/4) and variance (4 - pi)/4
    zscore = {}
    for pol in ['ee', 'nn']:
        abs_SNRs = np.array([np.abs(dly_filt_SNRs[bl]) for bl in bls_to_filter if bl[2] == pol])
        nbl = np.sum(np.isfinite(abs_SNRs), axis=0)
        with np.errstate(all='ignore'):
            zscore[pol] = np.where(nbl > 0, (np.nanmean(abs_SNRs, axis=0) - np.sqrt(np.pi / 4))
                                   / np.sqrt((4 - np.pi) / 4 / np.where(nbl > 0, nbl, 1)), np.nan)
    print(f'Finished cross-based RFI z-scores in {(time.time() - cross_rfi_start) / 60:.2f} minutes.')

In [ ]:
def data_vs_model_plot():
    if np.all([ant in overall_class.bad_ants for ant in ants]):
        print('All antennas classified as bad. Nothing to plot.')
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 6), dpi=100, sharex='col', sharey='row',
                             gridspec_kw={'hspace': 0, 'wspace': .05})
    for i, pol in enumerate(['ee', 'nn']):
        # the highest-redundancy modeled groups: the best short one and the best longer than 40 m
        group_bls = [bl for bl in red_avg_data if bl[0] != bl[1] and bl[2] == pol
                     and bl in model and not np.all(red_avg_flags[bl])]
        if len(group_bls) == 0:
            continue
        nsamp = {bl: np.median(red_avg_nsamples[bl]) for bl in group_bls}
        lengths = {bl: np.linalg.norm(hd.antpos[bl[1]] - hd.antpos[bl[0]]) for bl in group_bls}
        picks = [max((bl for bl in group_bls if lengths[bl] < 40), key=lambda bl: nsamp[bl], default=None),
                 max((bl for bl in group_bls if lengths[bl] >= 40), key=lambda bl: nsamp[bl], default=None)]
        for bl, color in zip(picks, ['cornflowerblue', 'sandybrown']):
            if bl is None:
                continue
            flagged = red_avg_flags[bl] | rfi_flags
            avg = np.where(flagged, np.nan, red_avg_data[bl])
            model_here = np.asarray(model[bl]).astype(complex)
            model_here = np.where(flagged | ~np.isfinite(model_here) | (model_here == 0), np.nan, model_here)
            # model as a thin, darker-shaded line drawn atop the data points
            darker = tuple(0.55 * channel for channel in matplotlib.colors.to_rgb(color))
            axes[1, i].semilogy(hd.freqs / 1e6, np.abs(avg[0]), '.', ms=2, color=color,
                                label=f'{(int(bl[0]), int(bl[1]), bl[2])} ({lengths[bl]:.0f} m, '
                                      f'{nsamp[bl]:.0f} effective baselines)')
            axes[1, i].semilogy(hd.freqs / 1e6, np.abs(model_here[0]), lw=.5, color=darker, zorder=3)
            axes[0, i].plot(hd.freqs / 1e6, np.angle(avg[0]), '.', ms=2, color=color)
            axes[0, i].plot(hd.freqs / 1e6, np.angle(model_here[0]), lw=.5, color=darker, zorder=3)
        model_handle = matplotlib.lines.Line2D([0], [0], color='k', lw=.75)
        handles, labels = axes[1, i].get_legend_handles_labels()
        axes[1, i].set_xlabel('Frequency (MHz)')
        axes[1, i].legend(handles=[model_handle] + handles, labels=['Sky Model'] + labels,
                          loc='upper right', title=f'{pol}-polarization')
    axes[0, 0].set_ylabel('Visibility Phase (radians)')
    axes[1, 0].set_ylabel('Visibility Amplitude' + (f' ({MODEL_VIS_UNITS})' if MODEL_VIS_UNITS else ''))
    plt.tight_layout()

# *Figure 6: Redundantly-averaged calibrated data compared to the sky model*

Redundantly-averaged calibrated visibilities (points) for the highest-redundancy short and intermediate-length groups per polarization (first integration), overlaid on their sky models (lines). Flagged channels are blanked. Good calibration means the amplitudes track and the phases scatter tightly around the model's.

In [ ]:
if not all_flagged() and DO_SKY_CAL: data_vs_model_plot()

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    # flag z-score outliers and their watershed neighbors, then fold into the RFI mask
    cross_rfi_flags = np.zeros_like(rfi_flags)
    for pol in ['ee', 'nn']:
        cross_rfi_flags |= np.nan_to_num(zscore[pol]) > CROSS_RFI_Z_THRESH
    while True:
        nflags = np.sum(cross_rfi_flags)
        for pol in ['ee', 'nn']:
            cross_rfi_flags |= xrfi._ws_flag_waterfall(np.nan_to_num(zscore[pol]),
                                                    cross_rfi_flags | rfi_flags, CROSS_RFI_WS_Z_THRESH)
        if np.sum(cross_rfi_flags) == nflags:
            break
    n_new = np.sum(cross_rfi_flags & ~rfi_flags)
    rfi_flags |= cross_rfi_flags
    flag_mostly_flagged_bands(rfi_flags)
    print(f'Cross-based RFI flagging added {n_new} new (time, channel) flags '
          f'({np.mean(rfi_flags):.2%} of the waterfall now flagged).')

In [ ]:
def cross_rfi_zscore_plot():
    if zscore is None:
        print('Cross-based RFI z-scores were not computed. Nothing to plot.')
        return
    fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(14, 6), dpi=100, gridspec_kw={'hspace': 0})
    for ax, pol in zip(axes, ['ee', 'nn']):
        for tind, time_ in enumerate(data.times):
            color = f'C{tind}'
            ax.plot(hd.freqs / 1e6, zscore[pol][tind], alpha=.3, lw=.5, color=color,
                    label=f'JD {time_:.6f} (t={tind}, before cross-based flagging)')
            ax.plot(hd.freqs / 1e6, np.where(rfi_flags[tind], np.nan, zscore[pol][tind]), lw=.75, color=color,
                    label=f'JD {time_:.6f} (t={tind}, after cross-based flagging)')
        ax.axhline(CROSS_RFI_Z_THRESH, color='r', ls='--', lw=.5, label='Flagging Threshold')
        ax.axhline(CROSS_RFI_WS_Z_THRESH, color='darkorange', ls='--', lw=.5, label='Watershed Threshold')
        ax.set_ylabel(f'{pol} z-score')
    axes[0].legend(ncols=3)
    axes[1].set_xlabel('Frequency (MHz)')
    plt.tight_layout()

# *Figure 7: Delay-filtered redundantly-averaged z-scores before and after cross-based flagging*

Per-polarization z-scores from incoherently averaging |SNR| of the delay-filtered, redundantly-averaged calibrated visibilities — the same statistic as the `delay_filtered_average_zscore` notebook, computed here per file. Faint curves show the z-scores before cross-based flagging; solid curves have the new flags applied. Excursions above the flagging threshold were flagged, then grown outward through neighbors above the watershed threshold; the full z-score waterfalls are saved for whole-night synthesis by `full_day_rfi_round_2`.

In [ ]:
if not all_flagged() and DO_SKY_CAL: cross_rfi_zscore_plot()

In [ ]:
if SAVE_RESULTS and not all_flagged() and DO_SKY_CAL:
    # save z-scores as a UVFlag metrics file for full-day synthesis by full_day_rfi_round_2
    uvd_meta = UVData()
    uvd_meta.read(SUM_FILE, read_data=False)
    uvf = UVFlag(uvd_meta, waterfall=True, mode='metric')
    uvf.select(polarizations=['ee', 'nn'])
    x_orientation = uvf.telescope.get_x_orientation_from_feeds()
    for pol in ['ee', 'nn']:
        pol_ind = np.argwhere(uvf.polarization_array == utils.polstr2num(pol, x_orientation=x_orientation))[0][0]
        uvf.metric_array[:, :, pol_ind] = zscore[pol]
    uvf.history += ('Produced by file_sky_calibration notebook with the following environment:\n'
                    + '=' * 65 + '\n' + os.popen('conda env export').read() + '=' * 65)
    uvf.write(RED_AVG_ZSCORE_FILE, clobber=True)
    del uvd_meta, uvf
    malloc_trim()
    print(f'Wrote cross-based RFI z-scores to {RED_AVG_ZSCORE_FILE}.')

### Relative phase calibration between polarizations

Sky calibration ties each polarization's phases to the model independently, so the one remaining polarization degree of freedom is the global relative phase $\Delta = \phi_{ee} - \phi_{nn}$. It is solved from cross-polarized (en/ne) visibilities against the model with `abscal.cross_pol_phase_cal`, which by construction affects only that single degree of freedom and not any per-antenna phase. This requires the sky model to include cross-polarized visibilities; if it does not, this step is skipped.

In [ ]:
delta = None
if not all_flagged() and DO_SKY_CAL and CALIBRATE_CROSS_POLS:
    if not all(pol in pols_to_load for pol in ('en', 'ne')):
        print('Sky model has no cross-polarized visibilities, so relative phase calibration is being skipped.')
    else:
        cross_pol_cal_start = time.time()
        # per-antpol flags for building weighted redundant averages of calibrated cross-pols
        gain_flags = {ant: ~np.isfinite(g) | rfi_flags for ant, g in gains.items()}

        cross_reds = redcal.get_reds(hd.antpos, pols=['en', 'ne'])
        cross_reds = redcal.filter_reds(cross_reds, ex_ants=overall_class.bad_ants, antpos=hd.antpos,
                                        min_bl_cut=SKYCAL_MIN_BL_LEN, max_bl_cut=SKYCAL_MAX_BL_LEN)
        data_here, wgts_here, cross_bls = {}, {}, []
        for red in cross_reds:
            red_ok = [bl for bl in red if bl in data
                      and utils.split_bl(bl)[0] in gains and utils.split_bl(bl)[1] in gains]
            if len(red_ok) == 0 or red_ok[0] not in model:
                continue
            calibrated, wgt = [], 0
            for bl in red_ok:
                ant_i, ant_j = utils.split_bl(bl)
                fl = gain_flags[ant_i] | gain_flags[ant_j]
                calibrated.append(np.where(fl, np.nan, data[bl] / (gains[ant_i] * np.conj(gains[ant_j]))))
                wgt = wgt + (~fl)
            data_here[red_ok[0]] = np.nanmean(calibrated, axis=0)
            wgts_here[red_ok[0]] = wgt
            cross_bls.append(red_ok[0])

        # \Delta = \phi_e - \phi_n, where V_{en}^{cal} = V_{en}^{uncal} * e^{i \Delta}
        # and V_{ne}^{cal} = V_{ne}^{uncal} * e^{-i \Delta}
        delta = abscal.cross_pol_phase_cal(model=model, data=data_here, wgts=wgts_here,
                                           data_bls=cross_bls, model_bls=cross_bls,
                                           return_gains=False, refpol='Jee')
        for ant in gains:
            if ant[1] == 'Jnn':
                gains[ant] = gains[ant] * np.exp(1j * delta)
        print(f'Finished relative polarized phase calibration in '
              f'{(time.time() - cross_pol_cal_start) / 60:.2f} minutes.')

In [ ]:
def polarized_gain_phase_plot():
    if delta is None:
        print('Relative phase calibration was not performed. Nothing to plot.')
        return
    plt.figure(figsize=(14, 4), dpi=100)
    for i, time_ in enumerate(data.times):
        plt.plot(data.freqs / 1e6, np.where(rfi_flags[i], np.nan, delta[i, :]), '.', ms=1.5, label=f'JD {time_:.6f} (t={i})')
    plt.ylim(-np.pi - 0.5, np.pi + 0.5)
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Relative Phase $\\Delta \\ (\\phi_{ee} - \\phi_{nn})$')
    plt.grid()
    plt.legend()

# *Figure 8: Relative Phase Calibration*

This figure shows the relative phase calibration between the `ee` vs. `nn` polarizations.

In [ ]:
if not all_flagged() and DO_SKY_CAL: polarized_gain_phase_plot()

### Spectator calibration of excluded antennas

Antennas excluded from calibration would otherwise carry no gains or $\chi^2$, leaving downstream full-day antenna flagging without comparable statistics for every antenna in every file (and biasing day-level metrics toward the files where a marginal antenna happened to pass). Here every excluded antenna is calibrated as a "spectator" against the frozen solution by `skycal.expand_sky_gains`: with the good antennas' gains and the sky model held fixed, each spectator gain is a closed-form per-channel weighted least-squares over its inter-SNAP baselines to good antennas — no iteration, and by construction unable to affect any other antenna's solution. Its sky-model $\chi^2$ is computed against the same frozen reference and attributed only to the spectator. Broken antennas get garbage gains with huge $\chi^2$; they remain flagged, and the statistic records how bad they are. Spectator gains are excluded from the decoherence estimate and from the redundant averages.

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    spectator_start = time.time()
    # candidate baselines subject to the same length cuts as the main solve; expand_sky_gains keeps
    # only inter-SNAP baselines (given ant_to_SNAP_dict) with exactly one solved antenna, orients
    # them spectator-first, and solves each spectator against the frozen gains and the sky model,
    # updating gains and cspa in place
    cand_bls = [bl for bl in data if bl[0] != bl[1] and bl[2] in ('ee', 'nn') and bl in model
                and SKYCAL_MIN_BL_LEN <= np.linalg.norm(hd.antpos[bl[1]] - hd.antpos[bl[0]]) <= SKYCAL_MAX_BL_LEN]
    solved_ants = set(gains)
    skycal.expand_sky_gains(data, model, gains, autos=autos, model_flags=model_flags,
                            ant_flags={ant: rfi_flags for bl in cand_bls for ant in utils.split_bl(bl)},
                            bls=cand_bls, ant_to_SNAP_dict=ant_snap, dt=int_time, df=chan_res,
                            chisq_per_ant=cspa)
    spectators = sorted(set(gains) - solved_ants)
    if len(spectators) == 0:
        print('No excluded antennas with qualifying baselines; no spectator gains to fill in.')
    else:
        print(f'Filled in spectator gains and chi^2 for {len(spectators)} excluded antpols '
              f'in {(time.time() - spectator_start) / 60:.2f} minutes.')


## Estimate per-SNAP, per-X-engine-block decoherence

Missed F-engine → X-engine packets cause the correlator to reuse stale data, which suppresses cross-correlations between antennas on *different* SNAPs by $(1 - p_i)(1 - p_j)$ while leaving autocorrelations and intra-SNAP baselines untouched. Because the starting gain amplitudes come from the (exempt) autocorrelations, that suppression lands cleanly in the refined gains, where `skycal.estimate_SNAP_decoherence` fits it as a per-SNAP staircase in X-engine blocks on top of a smooth bandpass. Note that within each band, the fit is relative to that band's least-suppressed block — absolute levels are not measured. Results are saved to a `SNAPDecoherence` sidecar file for downstream correction, and each antenna is classified by two metrics of its SNAP's decoherence: the worst loss fraction anywhere, and the largest change between adjacent X-engine blocks within a band (since $p$ is relative to each band's cleanest block, changes across the band split at FM are not counted). The sidecar freezes the antenna → SNAP mapping *as used here* — including any identity repairs, so it may deliberately disagree with M&C. Downstream corrections must use the stored map (never a fresh M&C walk) and must apply the same identity relabeling to any raw data first.

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    deco_start = time.time()
    # final decoherence estimate: because every channel is calibrated independently, the
    # cross-based RFI flags cannot change any surviving channel's gains, so no re-solve is needed --
    # they enter here by zeroing the newly-flagged channels' weights in the block fits
    logamp_wgts = skycal.log_gain_inverse_variance(meta['wgts'], meta['g0'], meta['refined_gains'], ant_snap)
    logamp_wgts = {ant: np.where(rfi_flags, 0, wgt) for ant, wgt in logamp_wgts.items()}
    # spectator gains are excluded: only antennas solved in the main loop have logamp_wgts
    deco, dmeta = skycal.estimate_SNAP_decoherence({ant: gains[ant] for ant in logamp_wgts},
                                                   logamp_wgts, ant_snap, np.asarray(hd.freqs))
    sd = io.SNAPDecoherence.from_estimate(
        deco, dmeta, ant_snap, np.asarray(hd.freqs), np.asarray(hd.times),
        estimator_kwargs=dict(nchans_per_block=96, gain_smoothing_scale=100e-9, eigenval_cutoff=1e-12,
                              detection_sigma=2.0, full_sigma=3.0, band_split_freq=100e6),
        history='Produced by file_sky_calibration notebook.')
    n_detected = sum(int(np.sum(np.nan_to_num(p) > 0)) for p in deco.values())
    print(f'Finished final decoherence estimation in {(time.time() - deco_start) / 60:.2f} minutes: '
          f'{n_detected} detected (SNAP, time, block) cells across {len(deco)} SNAPs.')

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    # classify each antenna by the decoherence of its SNAP (over times and blocks) with two
    # metrics: the worst absolute loss fraction p, and the largest change in p between
    # adjacent X-engine blocks within a band (p is relative to each band's cleanest block,
    # so changes across the band split at FM are not counted). Antennas whose SNAPs have
    # no measured blocks are left out of these classifications.
    def worst_deco_jump(p):
        '''Largest |change in p| between adjacent X-engine blocks within a band, over times.'''
        diffs = np.concatenate([np.abs(np.diff(p[:, sl], axis=1)).ravel() for sl in dmeta['band_slices']])
        return np.max(diffs[np.isfinite(diffs)]) if np.any(np.isfinite(diffs)) else np.nan

    worst_by_ant, jump_by_ant = {}, {}
    for ant in ants:
        if ant[0] in ant_snap and ant_snap[ant[0]] in sd.decoherence:
            p = sd.decoherence[ant_snap[ant[0]]]
            finite = p[np.isfinite(p)]
            if len(finite) > 0:
                worst_by_ant[ant] = np.max(finite)
            jump = worst_deco_jump(p)
            if np.isfinite(jump):
                jump_by_ant[ant] = jump
    deco_class = ant_class.antenna_bounds_checker(worst_by_ant, good=deco_max_good, suspect=deco_max_suspect,
                                              bad=[(-np.inf, np.inf)])
    deco_jump_class = ant_class.antenna_bounds_checker(jump_by_ant, good=deco_jump_good, suspect=deco_jump_suspect,
                                                   bad=[(-np.inf, np.inf)])
    overall_class += deco_class
    overall_class += deco_jump_class
    for SNAP in sorted(sd.decoherence, key=snap_sort_key):
        p = sd.decoherence[SNAP]
        finite = p[np.isfinite(p)]
        if len(finite) > 0 and np.max(finite) > deco_max_suspect[1]:
            print(f'{snap_label(SNAP)} is severely decoherent (worst p = {np.max(finite):.2%} '
                  f'> {deco_max_suspect[1]:.0%}); its antennas are classified as bad.')
        jump = worst_deco_jump(p)
        if np.isfinite(jump) and jump > deco_jump_suspect[1]:
            print(f'{snap_label(SNAP)} has a large decoherence jump between adjacent X-engine blocks '
                  f'(max = {jump:.2%} > {deco_jump_suspect[1]:.0%}); its antennas are classified as bad.')

In [ ]:
def decoherence_matrix_plot():
    SNAPs = sorted(sd.SNAPs, key=snap_sort_key)
    M = 100 * np.array([sd.decoherence[SNAP] for SNAP in SNAPs])   # (NSNAP, Ntimes, Nblocks)
    ntimes = M.shape[1]
    cmap = plt.get_cmap('Reds').copy()
    cmap.set_bad('0.85')
    finite_pos = M[np.isfinite(M) & (M > 0)]
    vmax = max(1.0, (np.percentile(finite_pos, 98) if len(finite_pos) else 1.0))

    fig, axes = plt.subplots(1, ntimes, figsize=(7.5 * ntimes + 1, 0.24 * len(SNAPs) + 3),
                             sharey=True, constrained_layout=True, squeeze=False)
    block_centers = np.mean(sd.block_freqs, axis=1) / 1e6
    for tind, ax in enumerate(axes[0]):
        im = ax.imshow(np.ma.masked_invalid(M[:, tind, :]), aspect='auto', cmap=cmap,
                       vmin=0, vmax=vmax, interpolation='none')
        ax.set_xticks(range(sd.block_freqs.shape[0]))
        ax.set_xticklabels([f'{fc:.1f}' for fc in block_centers], fontsize=6, rotation=90)
        prev = None
        for k, SNAP in enumerate(SNAPs):
            node = snap_node_slot(SNAP)[0]
            if prev is not None and node != prev:
                ax.axhline(k - 0.5, color='k', lw=0.7)
            prev = node
        ax.set_title(f'{data.times[tind]:.6f} (t={tind})')
        ax.set_xlabel('X-engine block center frequency (MHz)')
    axes[0, 0].set_yticks(range(len(SNAPs)))
    axes[0, 0].set_yticklabels([snap_label(SNAP) for SNAP in SNAPs], fontsize=6)
    # SNAPs whose antennas are flagged for exceeding either decoherence limit get red labels
    bad_SNAPs = {ant_snap[ant[0]] for cls in [deco_class, deco_jump_class]
                 for ant in cls.bad_ants if ant[0] in ant_snap}
    for tick_label, SNAP in zip(axes[0, 0].get_yticklabels(), SNAPs):
        if SNAP in bad_SNAPs:
            tick_label.set_color('r')
    fig.colorbar(im, ax=axes, shrink=0.7, label='Decoherence $\\hat{p}$ (%)')

# *Figure 9: Per-SNAP, per-X-engine-block decoherence*

The fitted loss fraction $p$ per (SNAP, X-engine block), one panel per integration (states change discontinuously in time, so integrations are fit independently). Gray cells were not measurable (e.g. fully-flagged blocks like the FM band). White means no detected suppression at 2$\sigma$. Within each band (split around 100 MHz), values are relative to that band's least-suppressed block. SNAP labels are shown in red if their antennas are flagged for exceeding either decoherence limit (worst loss fraction or largest adjacent-block jump); the classification uses the worst value over all times and blocks, so a SNAP flagged in any integration is flagged for the whole file.

In [ ]:
if not all_flagged() and DO_SKY_CAL: decoherence_matrix_plot()

In [ ]:
def staircase_plot(nshow=6):
    scores = sorted([(np.nanmax(sd.decoherence[SNAP][tind]), SNAP, tind)
                     for SNAP in sd.SNAPs for tind in range(len(data.times))
                     if np.any(np.nan_to_num(sd.decoherence[SNAP][tind]) > 0)], reverse=True)
    if len(scores) == 0:
        print('No decoherence detected anywhere. Nothing to plot.')
        return
    picks = scores[:nshow]
    chan_to_block = dmeta['chan_to_block']

    fig, axes = plt.subplots(len(picks), 1, figsize=(14, 2.8 * len(picks)), dpi=100,
                             sharex=True, constrained_layout=True, squeeze=False)
    for ax, (worst, SNAP, tind) in zip(axes[:, 0], picks):
        ys, ws = [], []
        for ant in sorted(gains):
            if ant not in logamp_wgts or ant_snap.get(ant[0]) != SNAP:
                continue
            w = logamp_wgts[ant][tind]
            if not (w > 0).any():
                continue
            with np.errstate(invalid='ignore', divide='ignore'):
                y = np.log(np.abs(gains[ant][tind]))
            y = np.where((w > 0) & np.isfinite(y), y, np.nan)
            w = np.where(np.isfinite(y), w, 0)
            y = y - np.nansum(y * w) / w.sum()
            ax.plot(hd.freqs / 1e6, y, color='0.75', lw=0.5)
            ys.append(np.nan_to_num(y) * w)
            ws.append(w)
        wsum = np.sum(ws, axis=0)
        with np.errstate(invalid='ignore', divide='ignore'):
            mean_y = np.where(wsum > 0, np.sum(ys, axis=0) / np.maximum(wsum, 1e-30), np.nan)
        ax.plot(hd.freqs / 1e6, mean_y, 'k', lw=1.2, label='mean ln|g|')

        # overlay the fitted staircase, aligned to the data by a weighted vertical offset and
        # drawn one X-engine block at a time (no lines connecting different blocks)
        stair = -np.nan_to_num(dmeta['log_suppression'][SNAP][tind])[chan_to_block]
        good = wsum > 0
        offset = np.sum((mean_y - stair)[good] * wsum[good]) / wsum[good].sum()
        labeled = set()
        for b in range(sd.block_freqs.shape[0]):
            p_b = sd.decoherence[SNAP][tind, b]
            if not np.isfinite(p_b):
                continue  # unmeasured block: nothing was fit
            chans = chan_to_block == b
            color, label = (('r', 'fitted staircase $-\\ln(1 - \\hat{p})$') if p_b > 0
                            else ('b', 'no decoherence detected'))
            ax.plot(hd.freqs[chans] / 1e6, np.where(good[chans], stair[chans] + offset, np.nan),
                    color=color, lw=1.5, label=(None if label in labeled else label))
            labeled.add(label)
        for b in range(1, sd.block_freqs.shape[0]):
            ax.axvline(sd.block_freqs[b, 0] / 1e6, color='r', ls=':', lw=0.5, alpha=0.5)

        ax.set_title(f'{snap_label(SNAP)}, JD {data.times[tind]:.6f} (t={tind})', fontsize=10)
        ax.legend(fontsize=8, loc='lower right')
        ax.set_ylabel('ln|g| (mean removed)')
    axes[-1, 0].set_xlabel('Frequency (MHz)')

# *Figure 10: Gain spectra of the most decoherent SNAPs*

For the most decoherent (SNAP, integration) fits: each of the SNAP's antenna-pol log-gain amplitude spectra (gray, per-spectrum weighted mean removed), their weighted mean (black), and the fitted per-block staircase (vertically aligned to the data), drawn one X-engine block at a time — red where suppression was detected, blue where the fit is consistent with no decoherence. Real decoherence steps at X-engine block walls (dotted red lines); smooth drift through a wall would indicate bandpass structure the staircase should not have absorbed.

In [ ]:
if not all_flagged() and DO_SKY_CAL: staircase_plot()

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    # redundantly average the final calibrated, decoherence-corrected data, accumulating
    # redundant-baseline chi^2 (a model-independent cross-check on the calibration) as we go
    red_avg_data, red_avg_flags, red_avg_nsamples, red_avg_meta = apply_cal.calibrate_and_red_avg(
        data, gains, red_avg_reds, ant_flags={ant: rfi_flags for ant in gains},
        ex_ants=overall_class.bad_ants, snap_decoherence=sd, dt=int_time, df=chan_res)
    redcal_cspa, redcal_chisq = red_avg_meta['chisq_per_ant'], red_avg_meta['total_chisq']
    for jpol in sorted(redcal_chisq):
        unflagged = np.where(rfi_flags, np.nan, redcal_chisq[jpol])
        print(f'{jpol} redundant-baseline chi^2 per DoF (unflagged): mean = {np.nanmean(unflagged):.3f}, '
              f'median = {np.nanmedian(unflagged):.3f}')


In [ ]:
def chisq_array_plot(chisq_per_ant, statistic, metric='mean', vmax=None):
    '''Array plot of a per-antenna chi^2 statistic, averaged over unflagged times and channels
    with the given metric ('mean' or 'median'), stated explicitly in the panel titles.
    vmax=None picks a robust scale from the data.'''
    if chisq_per_ant is None or np.all([ant in overall_class.bad_ants for ant in ants]):
        print('Nothing to plot.')
        return
    avg_alg = (np.nanmedian if metric == 'median' else np.nanmean)
    avgs = {ant: avg_alg(np.where(rfi_flags, np.nan, chisq_per_ant[ant])) for ant in chisq_per_ant}
    if vmax is None:
        vmax = max(2, np.nanpercentile([m for m in avgs.values() if np.isfinite(m)], 90))

    def _chisq_subplot(antnums, size=250):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)
        for ax, pol in zip(axes, ['ee', 'nn']):
            # invisible scatter of every antenna position so flagged (chi^2-less) antennas
            # still fall inside the axes limits
            ax.scatter([hd.antpos[antnum][0] for antnum in antnums],
                       [hd.antpos[antnum][1] for antnum in antnums], s=size, facecolors='none', edgecolors='none')
            ants_here = [(antnum, utils.split_pol(pol)[0]) for antnum in antnums]
            finite = [ant for ant in ants_here if np.isfinite(avgs.get(ant, np.nan))]
            scatter = ax.scatter([hd.antpos[ant[0]][0] for ant in finite],
                                 [hd.antpos[ant[0]][1] for ant in finite],
                                 s=size, c=[avgs[ant] for ant in finite], lw=.25, edgecolors='none',
                                 norm=matplotlib.colors.LogNorm(vmin=1, vmax=vmax))
            for ant in ants_here:
                in_scatter = np.isfinite(avgs.get(ant, np.nan))
                ax.text(hd.antpos[ant[0]][0], hd.antpos[ant[0]][1], ant[0], va='center', ha='center', fontsize=8,
                        c=('w' if in_scatter and ant not in overall_class.bad_ants else 'r'))
            plt.colorbar(scatter, ax=ax, extend='both')
            ax.axis('equal')
            ax.set_xlabel('East-West Position (meters)')
            ax.set_ylabel('North-South Position (meters)')
            ax.set_title(f'{pol}-pol {metric.capitalize()} {statistic} / Antenna')
        plt.tight_layout()

    _chisq_subplot([antnum for antnum in hd.data_ants if antnum < 320])
    outriggers = [antnum for antnum in hd.data_ants if antnum >= 320 and any(ant[0] == antnum for ant in avgs)]
    if len(outriggers) > 0:
        _chisq_subplot(outriggers, size=400)

# *Figure 11: Sky-model chi^2 per antenna across the array*

This plot shows the mean (taken over time and frequency, excluding flagged channels; the statistic is stated in the panel titles) of the normalized $\chi^2$ per antenna against the sky model. The expectation for noise-like residuals is 1.0, plus a common floor from sky-model error. Flagged antennas (numbered in red) carry the $\chi^2$ of their spectator solve against the frozen good-antenna solution. The color scale is fixed relative to the bad threshold, so saturated circles mark antennas far above it.

In [ ]:
if not all_flagged() and DO_SKY_CAL: chisq_array_plot(cspa, 'Sky-Model $\\chi^2$', vmax=sc_cspa_suspect[1])

# *Figure 12: Redundant-baseline chi^2 per antenna across the array*

The mean (over unflagged times and frequencies; the statistic is stated in the panel titles) of the DoF-normalized weighted scatter of the final calibrated, decoherence-corrected visibilities about their redundant-group means. Unlike the sky-model $\chi^2$ in Figure 11, this is model-independent — and unlike omnical's $\chi^2$, it is **not** expected to be ~1: sky calibration does not fit the gains to minimize non-redundancy, so this statistic measures the array's intrinsic non-redundancy (position errors, beam-to-beam variation) plus any calibration error, in units of thermal noise. Values of a few are typical; what matters is *relative* structure — antennas standing out from their neighbors. Flagged antennas (numbered in red) are measured against the good-antenna group means, attributed only to themselves and excluded from the averages; flagged antennas with no usable data show no value.

In [ ]:
if not all_flagged() and DO_SKY_CAL: chisq_array_plot(redcal_cspa, 'Redundant-Baseline $\\chi^2$', vmax=sc_cspa_suspect[1])

# *Figure 13: Summary of antenna classifications after sky calibration*

This figure is the same as [Figure 3](#Figure-3:-Summary-of-antenna-classifications-prior-to-calibration), except that it now includes additional suspect or bad antennas based on sky calibration $\chi^2$ and on the decoherence of each antenna's SNAP (see above).

In [ ]:
if not all_flagged() and DO_SKY_CAL: array_class_plot(overall_class, extra_label=', Post-Sky-Cal')

In [ ]:
to_show = {'Antenna': [f'{ant[0]}{ant[1][-1]}' for ant in ants]}
classes = {'Antenna': [overall_class[ant] if ant in overall_class else '-' for ant in ants]}
to_show['Dead?'] = [{'good': 'No', 'bad': 'Yes'}[am_totally_dead[ant]] if (ant in am_totally_dead) else '' for ant in ants]
classes['Dead?'] = [am_totally_dead[ant] if (ant in am_totally_dead) else '' for ant in ants]
for title, ac in [('Low Correlation', am_corr),
                  ('Cross-Polarized', am_xpol),
                  ('Solar Alt', solar_class),
                  ('Even/Odd Zeros', zeros_class),
                  ('Autocorr Power', auto_power_class),
                  ('Autocorr Slope', auto_slope_class),
                  ('Auto RFI RMS', auto_rfi_class),
                  ('Autocorr Shape', auto_shape_class),
                  ('Bad Diff X-Engines', xengine_diff_class),
                  ('Identity Coherence', identity_class)]:
    to_show[title] = [f'{ac._data[ant]:.2G}' if (ac is not None and ant in ac._data) else '' for ant in ants]
    classes[title] = [ac[ant] if (ac is not None and ant in ac) else 'bad' for ant in ants]

to_show['Sky Cal chi^2'] = [f'{np.nanmedian(np.where(rfi_flags, np.nan, cspa[ant])):.3G}' \
                            if (cspa is not None and ant in cspa) else '' for ant in ants]
classes['Sky Cal chi^2'] = [skycal_class[ant] if skycal_class is not None and ant in skycal_class else 'bad' for ant in ants]

to_show['Worst SNAP Decoherence'] = [f'{100 * deco_class._data[ant]:.2f}%' \
                                     if (deco_class is not None and ant in deco_class._data) else '' for ant in ants]
classes['Worst SNAP Decoherence'] = [deco_class[ant] if deco_class is not None and ant in deco_class else '' for ant in ants]

to_show['Worst Decoherence Jump'] = [f'{100 * deco_jump_class._data[ant]:.2f}%' \
                                     if (deco_jump_class is not None and ant in deco_jump_class._data) else '' for ant in ants]
classes['Worst Decoherence Jump'] = [deco_jump_class[ant] if deco_jump_class is not None and ant in deco_jump_class else '' for ant in ants]

df = pd.DataFrame(to_show)
df_classes = pd.DataFrame(classes)
colors = {'good': 'darkgreen', 'suspect': 'goldenrod', 'bad': 'maroon'}
df_colors = df_classes.map(lambda x: f'background-color: {colors.get(x, None)}')

table = df.style.hide() \
                .apply(lambda x: pd.DataFrame(df_colors.values, columns=x.columns), axis=None) \
                .set_properties(subset=['Antenna'], **{'font-weight': 'bold', 'border-right': "3pt solid black"}) \
                .set_properties(subset=df.columns[1:], **{'border-left': "1pt solid black"}) \
                .set_properties(**{'text-align': 'center', 'color': 'white'})

# *Table 1: Complete summary of per-antenna classifications*

This table summarizes the results of the various classifications schemes detailed above.
As before, <font color='#006400'>green is good</font>, <font color='#DAA520'>yellow is suspect</font>, and <font color='#800000'>red is bad</font>. The color for each antenna (first column) is the final summary of all other classifications.
Antennas missing from the sky calibration $\chi^2$ column were excluded from sky calibration, either because they were flagged earlier for some other reason or because they lack an M&C SNAP mapping. Antennas missing from the decoherence column are on SNAPs with no measurable blocks (uncolored, not bad).

In [ ]:
HTML(table.to_html())

In [ ]:
# Save antenna classification table as a csv
if SAVE_RESULTS:
    for ind, col in zip(np.arange(len(df.columns), 0, -1), df_classes.columns[::-1]):
        df.insert(int(ind), col + ' Class', df_classes[col])
    df.to_csv(ANTCLASS_FILE)

In [ ]:
print('Final Ant-Pol Classification:\n\n', overall_class)

## Save calibration solutions

In [ ]:
if not all_flagged() and DO_SKY_CAL:
    # build per-antpol flags; fill in unit gains for antennas that were never calibrated
    sky_flags = {}
    for ant in ants:
        if ant in gains:
            sky_flags[ant] = ~np.isfinite(gains[ant]) | (ant in overall_class.bad_ants) | rfi_flags
            gains[ant] = np.where(np.isfinite(gains[ant]), gains[ant], 1.0 + 0.0j)
        else:
            gains[ant] = np.ones((len(hd.times), len(hd.freqs)), dtype=np.complex64)
            sky_flags[ant] = np.ones((len(hd.times), len(hd.freqs)), dtype=bool)

    # for flagged channels outside the FM gap, replace the gain values with a smooth DPSS
    # model fit to the unflagged channels (the flags themselves are KEPT), so downstream
    # smoothing and inspection see plausible values rather than placeholders. The 100 ns
    # half-width matches the decoherence estimator's gain smoothing scale.
    cache = {}
    for ant in gains:
        wgt = (~sky_flags[ant]).astype(float)
        if not wgt.any():
            continue
        for band in [low_band, high_band]:
            rows_ok = wgt[:, band].sum(axis=1) > 0
            fill = sky_flags[ant][:, band] & rows_ok[:, None]
            if not fill.any():
                continue
            with np.errstate(all='ignore'):
                smooth_model, _, _ = dspec.fourier_filter(hd.freqs[band], gains[ant][:, band],
                                                          wgts=wgt[:, band], filter_centers=[0],
                                                          filter_half_widths=[100e-9], mode='dpss_solve',
                                                          suppression_factors=[1e-9], eigenval_cutoff=[1e-9],
                                                          max_contiguous_edge_flags=len(hd.freqs), cache=cache)
            gains[ant][:, band] = np.where(fill, smooth_model, gains[ant][:, band])
    del cache
    malloc_trim()

In [ ]:
if SAVE_RESULTS:
    add_to_history = 'Produced by file_sky_calibration notebook with the following environment:\n' + '=' * 65 + '\n' + os.popen('conda env export').read() + '=' * 65
    if len(identity_repair_note) > 0:
        add_to_history = identity_repair_note + '\n' + add_to_history

    if not all_flagged() and DO_SKY_CAL:
        hd_writer = io.HERAData(SUM_FILE)
        hc_sky = hd_writer.init_HERACal(gain_convention='divide', cal_style='sky')
        if MODEL_POL_CONVENTION is not None:
            hc_sky.pol_convention = MODEL_POL_CONVENTION
        if MODEL_VIS_UNITS is not None:
            hc_sky.gain_scale = MODEL_VIS_UNITS
        hc_sky.update(gains=gains, flags=sky_flags, quals=cspa, total_qual=total_chisq)
        if len(labeled_to_true) > 0:
            # machine-readable record of identity repairs: downstream consumers of RAW data
            # must apply this relabeling before using these gains or the decoherence sidecar
            hc_sky.extra_keywords['RELABELS'] = json.dumps({str(labeled): int(true_ant)
                                                            for labeled, true_ant in labeled_to_true.items()})
        hc_sky.history += add_to_history
        hc_sky.write_calfits(SKY_CAL_FILE, clobber=True)
        del hc_sky, hd_writer
        malloc_trim()

        if sd is not None:
            sd.history = add_to_history
            sd.write(DECOHERENCE_FILE, clobber=True)
            print(f'Wrote decoherence sidecar to {DECOHERENCE_FILE}.')

### Output fully-flagged placeholders for any product not written above

If the notebook bailed out early (e.g. every antenna classified as bad), the calibration, decoherence, and z-score products were never written. To keep the per-file output contract uniform — so that downstream consumers need no missing-file guards and the pipeline can treat an absent file as a failed job rather than a bad file — placeholders are written instead: a fully-flagged unit-gain calfits, an empty `SNAPDecoherence` sidecar (zero SNAPs, which multi-file reads merge as "unmeasured" at these times), and an all-nan z-score metrics file.

In [ ]:
if SAVE_RESULTS and not os.path.exists(SKY_CAL_FILE):
    print(f'WARNING: No calibration file produced at {SKY_CAL_FILE}. Creating a fully-flagged placeholder calibration file.')
    hd_writer = io.HERAData(SUM_FILE)
    # create fully flagged unit gains with chi^2 = 0
    hc_sky = hd_writer.init_HERACal(gain_convention='divide', cal_style='sky')
    hc_sky.history += add_to_history
    if MODEL_POL_CONVENTION is not None:
        hc_sky.pol_convention = MODEL_POL_CONVENTION
    if MODEL_VIS_UNITS is not None:
        hc_sky.gain_scale = MODEL_VIS_UNITS
    hc_sky.write_calfits(SKY_CAL_FILE, clobber=True)
    del hc_sky


if SAVE_RESULTS and not os.path.exists(DECOHERENCE_FILE):
    print(f'WARNING: No decoherence file produced at {DECOHERENCE_FILE}. Creating an empty placeholder sidecar.')
    sd_placeholder = io.SNAPDecoherence(
        decoherence={}, decoherence_refit={}, log_suppression_sigma={}, n_spectra_per_SNAP={},
        times=np.asarray(hd.times), block_freqs=np.asarray(hd.freqs).reshape(-1, 96),
        ant_to_SNAP_dict=(ant_snap if ant_snap is not None else {}),
        covered_blocks=np.zeros(len(hd.freqs) // 96, dtype=bool), edge_blocks=[],
        band_edges=np.full((1, 2), np.nan),
        history='Empty placeholder from file_sky_calibration (all antennas flagged; no decoherence measured).'
                + add_to_history)
    sd_placeholder.write(DECOHERENCE_FILE, clobber=True)
    del sd_placeholder

if SAVE_RESULTS and not os.path.exists(RED_AVG_ZSCORE_FILE):
    print(f'WARNING: No z-score file produced at {RED_AVG_ZSCORE_FILE}. Creating an all-nan placeholder metrics file.')
    uvd_meta = UVData()
    uvd_meta.read(SUM_FILE, read_data=False)
    uvf = UVFlag(uvd_meta, waterfall=True, mode='metric')
    uvf.select(polarizations=['ee', 'nn'])
    uvf.metric_array[:] = np.nan
    uvf.history += ('All-nan placeholder from file_sky_calibration (all antennas flagged; no z-scores computed).'
                    + add_to_history)
    uvf.write(RED_AVG_ZSCORE_FILE, clobber=True)
    del uvd_meta, uvf

## Metadata

In [ ]:
for repo in ['pyuvdata', 'hera_cal', 'hera_filters', 'hera_qm', 'hera_mc', 'hera_notebook_templates']:
    try:
        exec(f'from {repo} import __version__')
        print(f'{repo}: {__version__}')
    except ImportError:
        print(f'{repo}: not installed')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')